# Meta-control grid search and selected-model export

This notebook follows `rnn_data_prep.ipynb` and `rnn_training.ipynb`.

It:
1. loads the trained RNN and prepared testing caches from `~/Downloads/Meta-control`;
2. runs the same endpoint-error hybrid grid search;
3. uses inline Experiment 1 / Experiment 2 physics-rendering helpers (no standalone render scripts required);
4. saves the grid table; and
5. reruns the selected `A150_S20_e22` model and saves its full predictions, scene summary, and overlays for the final-figures notebook.


In [ ]:
# ============================================================
# 1. Imports, paths, checkpoint metadata, and grid settings
# ============================================================
from pathlib import Path
import json
import os
import sys
import time
import tempfile
import shutil
import itertools
from contextlib import nullcontext

# Run the entire grid in one process.
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "4")
os.environ.setdefault("SDL_VIDEODRIVER", "dummy")

import numpy as np
import pandas as pd
from PIL import Image
from scipy.stats import pearsonr
from IPython.display import display
import matplotlib.pyplot as plt

import torch
from torch import nn


HOME = Path.home()
BASE_DIR = HOME / "Downloads" / "Meta-control"


TEST_DATA_ROOT = BASE_DIR / "rnn_testing_data"
TEST_EXP_FOLDERS = ["exp1", "exp2"]

FRAME_CACHE_DIR = BASE_DIR / "compressed_frame_cache_100x128"

BALL_CACHE_DIR = BASE_DIR / "ball_position_cache"

HUMAN_SUMMARY_PATH = BASE_DIR / "data" / "scene_summary" / "empirical_scene_level_rt_accuracy_summary.csv"
SCENE_SUMMARY_PATH = BASE_DIR / "data" / "scene_summary" / "scene_summary.csv"

TRAINED_RUN_DIR = BASE_DIR / "rnn_training_results" / "lambda_0_2_offset0_everypoint_sliding15_nonfreeze_windows20_epochs150"

MODEL_PATH = (
    TRAINED_RUN_DIR
    / "best_int_checkpoint_rnn_model.pt"
)

PROJECT_ROOT = BASE_DIR / "physics_abstraction_master"

PROJECT_PYTHON = (
    PROJECT_ROOT
    / "python"
)



for required_path, label in [
    (TEST_DATA_ROOT, "testing-data root"),
    (MODEL_PATH, "trained RNN checkpoint"),
    (PROJECT_ROOT, "physics project"),
    (HUMAN_SUMMARY_PATH, "empirical human summary"),
    (SCENE_SUMMARY_PATH, "scene summary"),
]:
    if not required_path.exists():
        raise FileNotFoundError(
            f"Missing {label}: {required_path}"
        )


DEVICE = torch.device("cpu")
USE_AMP = False

TORCH_THREADS = min(
    4,
    os.cpu_count() or 1,
)

torch.set_num_threads(
    TORCH_THREADS
)

try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass


def autocast_context():
    return nullcontext()


try:
    checkpoint = torch.load(
        MODEL_PATH,
        map_location=DEVICE,
        weights_only=False,
    )
except TypeError:
    checkpoint = torch.load(
        MODEL_PATH,
        map_location=DEVICE,
    )


model_config = checkpoint.get(
    "config",
    {},
)

N_HISTORY = int(
    model_config.get(
        "N_HISTORY",
        15,
    )
)

CHECKPOINT_STRIDE = int(
    model_config.get(
        "CHECKPOINT_STRIDE",
        1,
    )
)

IMAGE_W = int(
    model_config.get(
        "IMAGE_W",
        100,
    )
)

IMAGE_H = int(
    model_config.get(
        "IMAGE_H",
        128,
    )
)

ORIGINAL_FRAME_WIDTH = float(
    model_config.get(
        "ORIGINAL_FRAME_WIDTH",
        800.0,
    )
)

ORIGINAL_FRAME_HEIGHT = float(
    model_config.get(
        "ORIGINAL_FRAME_HEIGHT",
        1024.0,
    )
)

COORD_SCALE = np.array(
    [
        ORIGINAL_FRAME_WIDTH,
        ORIGINAL_FRAME_HEIGHT,
    ],
    dtype=np.float32,
)

ERR_MAG_PIXEL_SCALE_FOR_PLOTS = float(
    np.mean(
        COORD_SCALE
    )
)

HIDDEN_CHANNELS = int(
    model_config.get(
        "HIDDEN_CHANNELS",
        96,
    )
)

TIME_EMBED_DIM = int(
    model_config.get(
        "TIME_EMBED_DIM",
        32,
    )
)

GLOBAL_MAX_FUTURE_OFFSET = int(
    model_config.get(
        "GLOBAL_MAX_FUTURE_OFFSET",
        1000,
    )
)


if N_HISTORY != 15:
    raise RuntimeError(
        "Expected the trained model to use "
        f"N_HISTORY=15; found {N_HISTORY}."
    )


GRID_OUTPUT_DIR = BASE_DIR / "meta_control_grid_search"
GRID_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

META_CONTROL_ROOT = BASE_DIR / "meta_control"
SELECTED_PARAMETER_LABEL = "A150_S20_e22"
SELECTED_A = 150
SELECTED_S = 20
SELECTED_E = 22.0

A_VALUES = [
    50,
    60,
    70,
    80,
    90,
    100,
    110,
    120,
    130,
    140,
    150,
]

S_VALUES = [
    20,
    30,
    40,
    50,
    60,
]

E_VALUES = [
    20.0,
    21.0,
    22.0,
    23.0,
    24.0,
]

PARAMETER_GRID = list(
    itertools.product(
        A_VALUES,
        S_VALUES,
        E_VALUES,
    )
)


# These globals are updated before evaluating each parameter combination serially.
ABSTRACTION_CHECK_CHUNK_FRAMES = int(
    A_VALUES[0]
)

SIM_CHUNK_FRAMES = int(
    S_VALUES[0]
)

ADAPTIVE_SIM_INCREMENT_FRAMES = int(
    S_VALUES[0]
)


MIN_ABSTRACTION_ACCEPT_FRAMES = 1
FINAL_ABSTRACTION_SINGLE_CHECK_BELOW_FRAMES = None

# Safety caps only; neither uses the recorded scene duration.
ADAPTIVE_SIM_MAX_FRAMES = 3000
MAX_TIME_ONLY_ROLLOUT_FRAMES = 3000
MAX_SEGMENTS_PER_SCENE = 200

FPS_FOR_VELOCITY = 60.0

GOAL_CONTACT_DISTANCE_PX = 20.0
GROUND_CONTACT_DISTANCE_PX = 40.0

# Grid search never builds or saves large frame/position caches.
ALLOW_BUILD_MISSING_TEST_CACHES = False


print("Device:", DEVICE)
print("Execution mode: serial")
print("PyTorch CPU threads:", TORCH_THREADS)
print("Checkpoint:", MODEL_PATH)
print(
    "Checkpoint epoch:",
    checkpoint.get(
        "epoch",
        "unknown",
    ),
)
print(
    "Best held-out loss:",
    checkpoint.get(
        "best_test_total_loss",
        "unknown",
    ),
)
print("N_HISTORY:", N_HISTORY)
print(
    "Parameter combinations:",
    len(PARAMETER_GRID),
)
print(
    "Goal threshold:",
    GOAL_CONTACT_DISTANCE_PX,
)
print(
    "Ground threshold:",
    GROUND_CONTACT_DISTANCE_PX,
)


In [ ]:
# ============================================================
# 2. Scene, cache, and JSON helpers
# ============================================================
def scene_kind(
    scene_dir,
):
    parts = [
        part.lower()
        for part in Path(
            scene_dir
        ).parts
    ]

    if (
        "exp1" in parts
        or "exp1more" in parts
    ):
        return "exp1"

    if (
        "exp2" in parts
        or "exp2more" in parts
    ):
        return "exp2"

    raise ValueError(
        "Cannot infer experiment kind from "
        f"scene path: {scene_dir}"
    )


def _count_png_frames(
    scene_dir,
):
    files = sorted(
        (
            Path(scene_dir)
            / "frames"
        ).glob(
            "frame_*.png"
        )
    )

    if not files:
        files = sorted(
            (
                Path(scene_dir)
                / "frames"
            ).glob(
                "*.png"
            )
        )

    return len(
        files
    )


def get_frame_files(
    scene_dir,
):
    files = sorted(
        (
            Path(scene_dir)
            / "frames"
        ).glob(
            "frame_*.png"
        )
    )

    if not files:
        files = sorted(
            (
                Path(scene_dir)
                / "frames"
            ).glob(
                "*.png"
            )
        )

    return files


def scene_rel_for_cache(
    scene_dir,
):
    path = Path(
        scene_dir
    )

    try:
        return (
            path.resolve()
            .relative_to(
                BASE_DIR.resolve()
            )
        )
    except Exception:
        return path


def safe_scene_id(
    scene_dir,
):
    relative = scene_rel_for_cache(
        scene_dir
    )

    return (
        str(relative)
        .replace(
            "/",
            "__",
        )
        .replace(
            "\\",
            "__",
        )
        .replace(
            ":",
            "",
        )
    )


def frame_cache_path(
    scene_dir,
):
    return (
        FRAME_CACHE_DIR
        / (
            f"{safe_scene_id(scene_dir)}"
            f"__frames_{IMAGE_H}x{IMAGE_W}"
            "_rgb_uint8.npy"
        )
    )


def ball_cache_path(
    scene_dir,
):
    return (
        BALL_CACHE_DIR
        / (
            f"{safe_scene_id(scene_dir)}"
            "__ball_positions_from_frames.csv"
        )
    )


def load_scene_frames_cache(
    scene_dir,
):
    path = frame_cache_path(
        scene_dir
    )

    if not path.exists():
        raise FileNotFoundError(
            "Grid search will not build missing "
            f"frame caches. Missing: {path}"
        )

    return np.load(
        path,
        mmap_mode="r",
    )


def load_scene_positions(
    scene_dir,
):
    path = ball_cache_path(
        scene_dir
    )

    if not path.exists():
        raise FileNotFoundError(
            "Grid search will not build missing "
            f"ball-position caches. Missing: {path}"
        )

    return pd.read_csv(
        path
    )


def find_testing_scene_dirs():
    scene_dirs = []

    for experiment in TEST_EXP_FOLDERS:
        experiment_dir = (
            TEST_DATA_ROOT
            / experiment
        )

        if not experiment_dir.exists():
            print(
                "Warning: missing testing folder:",
                experiment_dir,
            )
            continue

        for scene_dir in sorted(
            experiment_dir.iterdir()
        ):
            if (
                scene_dir.is_dir()
                and (
                    scene_dir
                    / "frames"
                ).exists()
                and _count_png_frames(
                    scene_dir
                )
                >= N_HISTORY + 1
            ):
                scene_dirs.append(
                    scene_dir
                )

    return scene_dirs


test_scene_dirs = (
    find_testing_scene_dirs()
)

if not test_scene_dirs:
    raise RuntimeError(
        "No usable testing scenes were found."
    )


print(
    "Testing scenes:",
    len(
        test_scene_dirs
    ),
)


In [ ]:
# ============================================================
# 3. Exact trained RNN architecture and strict checkpoint load
# ============================================================
class InTCheckpointRNN(nn.Module):
    def __init__(self, hidden_channels=96, time_embed_dim=32):
        super().__init__()

        self.hidden_channels = hidden_channels
        self.time_embed_dim = time_embed_dim

        self.input_conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=5, stride=2, padding=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, hidden_channels, kernel_size=3, stride=1, padding=1),
            nn.ReLU(inplace=True),
        )

        self.recurrent_conv = nn.Conv2d(
            hidden_channels,
            hidden_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            groups=hidden_channels,
            bias=False,
        )
        self.gate_conv = nn.Conv2d(
            hidden_channels * 2,
            hidden_channels,
            kernel_size=1,
        )
        self.hidden_bias = nn.Parameter(
            torch.zeros(1, hidden_channels, 1, 1)
        )

        self.attn_conv = nn.Conv2d(
            hidden_channels,
            1,
            kernel_size=1,
        )

        self.scene_mlp = nn.Sequential(
            nn.Linear(hidden_channels, 192),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.05),
            nn.Linear(192, 192),
            nn.ReLU(inplace=True),
        )

        self.time_mlp = nn.Sequential(
            nn.Linear(4, time_embed_dim),
            nn.ReLU(inplace=True),
            nn.Linear(time_embed_dim, time_embed_dim),
            nn.ReLU(inplace=True),
        )

        decoder_in = 192 + time_embed_dim
        self.checkpoint_decoder = nn.Sequential(
            nn.Linear(decoder_in, 192),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.05),
            nn.Linear(192, 96),
            nn.ReLU(inplace=True),
        )

        self.position_head = nn.Linear(96, 2)
        self.error_head = nn.Linear(96, 1)

    def _attention_pool(self, hidden):
        batch, channels, height, width = hidden.shape
        logits = self.attn_conv(hidden).view(
            batch,
            1,
            height * width,
        )
        weights = torch.softmax(logits, dim=-1)
        hidden_flat = hidden.view(
            batch,
            channels,
            height * width,
        )
        return (hidden_flat * weights).sum(dim=-1)

    def _time_features(self, checkpoint_times):
        # Exact training-notebook behavior.
        tau = checkpoint_times.clamp(min=0.0, max=1.5)
        return torch.stack(
            [
                tau,
                tau ** 2,
                torch.sin(np.pi * tau),
                torch.cos(np.pi * tau),
            ],
            dim=-1,
        )

    def forward(
        self,
        x,
        checkpoint_times,
        return_hidden=False,
    ):
        batch, n_frames, channels, height, width = x.shape
        hidden = None
        hidden_trace = []

        for frame_index in range(n_frames):
            z_t = self.input_conv(x[:, frame_index])
            if hidden is None:
                hidden = torch.zeros_like(z_t)

            candidate = torch.tanh(
                z_t
                + self.recurrent_conv(hidden)
                + self.hidden_bias
            )
            gate = torch.sigmoid(
                self.gate_conv(
                    torch.cat([z_t, hidden], dim=1)
                )
            )
            hidden = (
                gate * candidate
                + (1.0 - gate) * hidden
            )

            if return_hidden:
                hidden_trace.append(hidden)

        pooled = self._attention_pool(hidden)
        scene_code = self.scene_mlp(pooled)

        time_features = self._time_features(
            checkpoint_times
        )
        time_code = self.time_mlp(time_features)

        n_checkpoints = checkpoint_times.shape[1]
        expanded_scene_code = scene_code.unsqueeze(1).expand(
            batch,
            n_checkpoints,
            scene_code.shape[-1],
        )
        decoder_input = torch.cat(
            [expanded_scene_code, time_code],
            dim=-1,
        )
        decoded = self.checkpoint_decoder(
            decoder_input
        )

        predicted_positions = self.position_head(
            decoded
        )
        predicted_error_raw = self.error_head(
            decoded
        )

        if return_hidden:
            hidden_trace = torch.stack(
                hidden_trace,
                dim=1,
            )
            return (
                predicted_positions,
                predicted_error_raw,
                hidden_trace,
            )

        return predicted_positions, predicted_error_raw


def positive_error_magnitude(predicted_error_raw):
    return nn.functional.softplus(
        predicted_error_raw
    ).squeeze(-1)


def unnormalize_y(normalized_position):
    return normalized_position * COORD_SCALE


model = InTCheckpointRNN(
    hidden_channels=HIDDEN_CHANNELS,
    time_embed_dim=TIME_EMBED_DIM,
).to(DEVICE)

state_dict = checkpoint.get(
    "model_state_dict",
    checkpoint,
)

if state_dict and all(
    key.startswith("module.")
    for key in state_dict
):
    state_dict = {
        key.removeprefix("module."): value
        for key, value in state_dict.items()
    }

model.load_state_dict(
    state_dict,
    strict=True,
)
model.eval()

print(model)
print(
    "\nParameter count:",
    sum(parameter.numel() for parameter in model.parameters()),
)
print("Checkpoint loaded strictly.")
print("GLOBAL_MAX_FUTURE_OFFSET:", GLOBAL_MAX_FUTURE_OFFSET)


In [ ]:
# ============================================================
# Inline Exp. 1 / Exp. 2 physics rendering helpers
#
# This replaces the former dependency on standalone
# standalone Experiment 1 / Experiment 2 render scripts.
# The scene construction and drawing logic below is taken from
# those scripts and uses physics_abstraction_master directly.
# ============================================================
import math
from types import SimpleNamespace
from PIL import Image, ImageDraw


class _RenderConfig:
    """Adapter for project classes expecting config.model_dump()."""

    def __init__(self, **kwargs):
        self.kwargs = kwargs

    def model_dump(self):
        return dict(self.kwargs)


def _render_as_xy(value, label="position"):
    original = value

    while isinstance(value, list) and len(value) == 1:
        value = value[0]

    if isinstance(value, dict):
        for key in ("position", "pos", "point", "location"):
            if key in value:
                value = value[key]
                break

    while isinstance(value, list) and len(value) == 1:
        value = value[0]

    if (
        isinstance(value, (list, tuple))
        and len(value) >= 2
        and isinstance(value[0], (int, float))
        and isinstance(value[1], (int, float))
    ):
        return float(value[0]), float(value[1])

    raise ValueError(
        f"Could not parse {label} as x,y. Got: {original!r}"
    )


def _render_parse_screen_size(data):
    width, height = 800, 1000
    if "screen_size" in data:
        screen_size = data["screen_size"]
        while isinstance(screen_size, list) and len(screen_size) == 1:
            screen_size = screen_size[0]
        if isinstance(screen_size, list) and len(screen_size) >= 2:
            width, height = int(screen_size[0]), int(screen_size[1])
    return width, height


def _render_get_json_bottom_border(data):
    bottom_args = data.get("bottom_border_args")
    if bottom_args and isinstance(bottom_args, list) and len(bottom_args) >= 2:
        return (
            _render_as_xy(
                bottom_args[0],
                "bottom_border_args[0]",
            ),
            _render_as_xy(
                bottom_args[1],
                "bottom_border_args[1]",
            ),
            "json_bottom_border_args",
        )
    return (
        (10.0, 990.0),
        (790.0, 990.0),
        "default_bottom_border",
    )


def _render_get_json_side_border_length(data, bottom_y):
    plinko_args = data.get("plinko_border_args", [])
    try:
        while isinstance(plinko_args, list) and len(plinko_args) == 1:
            plinko_args = plinko_args[0]

        if isinstance(plinko_args, (int, float)):
            return int(plinko_args), "json_plinko_border_args"

        if (
            isinstance(plinko_args, list)
            and len(plinko_args) >= 1
            and isinstance(plinko_args[0], (int, float))
        ):
            return int(plinko_args[0]), "json_plinko_border_args"
    except Exception:
        pass

    return int(bottom_y + 10), "fallback_bottom_y_plus_10"


def _render_parse_container(container, idx):
    while (
        isinstance(container, list)
        and len(container) == 1
        and isinstance(container[0], dict)
    ):
        container = container[0]

    if isinstance(container, dict):
        pos = _render_as_xy(
            container,
            f"container_args[{idx}].position",
        )
        width = float(
            container.get(
                "width",
                container.get("w", 80),
            )
        )
        angle = float(container.get("angle", 0))
        length = float(
            container.get(
                "length",
                container.get("l", 0),
            )
        )
        return {
            "id": idx,
            "x": pos[0],
            "y": pos[1],
            "width": width,
            "length": length,
            "angle": angle,
            "raw": container,
        }

    if isinstance(container, list):
        pos = _render_as_xy(
            container[0],
            f"container_args[{idx}][0]",
        )
        width = float(container[1]) if len(container) > 1 else 80.0
        length = float(container[2]) if len(container) > 2 else 0.0
        angle = float(container[3]) if len(container) > 3 else 0.0
        return {
            "id": idx,
            "x": pos[0],
            "y": pos[1],
            "width": width,
            "length": length,
            "angle": angle,
            "raw": container,
        }

    raise ValueError(
        f"Unknown container format: {container!r}"
    )


def _render_get_line_args_from_json(data):
    for key in [
        "line_args",
        "lines_args",
        "line_arg",
        "lines",
        "Line_args",
        "Line",
    ]:
        if key in data:
            value = data[key]
            if value is None:
                return [], key
            if isinstance(value, list):
                return value, key
            return [value], key
    return [], None


def _render_parse_line(line_arg, idx):
    original = line_arg

    while isinstance(line_arg, list) and len(line_arg) == 1:
        line_arg = line_arg[0]

    if isinstance(line_arg, dict):
        if "point_a" in line_arg and "point_b" in line_arg:
            point_a = _render_as_xy(
                line_arg["point_a"],
                f"line_args[{idx}].point_a",
            )
            point_b = _render_as_xy(
                line_arg["point_b"],
                f"line_args[{idx}].point_b",
            )
        elif "a" in line_arg and "b" in line_arg:
            point_a = _render_as_xy(
                line_arg["a"],
                f"line_args[{idx}].a",
            )
            point_b = _render_as_xy(
                line_arg["b"],
                f"line_args[{idx}].b",
            )
        elif "p1" in line_arg and "p2" in line_arg:
            point_a = _render_as_xy(
                line_arg["p1"],
                f"line_args[{idx}].p1",
            )
            point_b = _render_as_xy(
                line_arg["p2"],
                f"line_args[{idx}].p2",
            )
        else:
            raise ValueError(
                f"Line dict missing endpoints: {original!r}"
            )

        angle = float(line_arg.get("angle", 0))
        return {
            "id": idx,
            "point_a": point_a,
            "point_b": point_b,
            "angle": angle,
            "raw": original,
        }

    if isinstance(line_arg, list):
        if (
            len(line_arg) >= 2
            and isinstance(
                line_arg[0],
                (list, tuple, dict),
            )
            and isinstance(
                line_arg[1],
                (list, tuple, dict),
            )
        ):
            point_a = _render_as_xy(
                line_arg[0],
                f"line_args[{idx}][0]",
            )
            point_b = _render_as_xy(
                line_arg[1],
                f"line_args[{idx}][1]",
            )
            angle = (
                float(line_arg[2])
                if (
                    len(line_arg) > 2
                    and isinstance(
                        line_arg[2],
                        (int, float),
                    )
                )
                else 0.0
            )
        elif len(line_arg) >= 4:
            point_a = (
                float(line_arg[0]),
                float(line_arg[1]),
            )
            point_b = (
                float(line_arg[2]),
                float(line_arg[3]),
            )
            angle = (
                float(line_arg[4])
                if (
                    len(line_arg) > 4
                    and isinstance(
                        line_arg[4],
                        (int, float),
                    )
                )
                else 0.0
            )
        else:
            raise ValueError(
                f"Line list format not understood: {original!r}"
            )

        return {
            "id": idx,
            "point_a": point_a,
            "point_b": point_b,
            "angle": angle,
            "raw": original,
        }

    raise ValueError(
        f"Unknown line format: {original!r}"
    )


class _InlineRenderModule:
    """Minimal module-like wrapper used by the hybrid rollout."""

    RENDER_WIDTH = 800
    RENDER_HEIGHT = 1024

    def __init__(self, experiment_kind):
        if experiment_kind not in {"exp1", "exp2"}:
            raise ValueError(experiment_kind)
        self.experiment_kind = experiment_kind

    def build_scene_from_json(
        self,
        json_path,
        log,
        goal_y_offset,
    ):
        import objects
        from scene import Scene

        json_path = Path(json_path)
        with open(json_path, "r") as file:
            data = json.load(file)

        json_width, json_height = (
            _render_parse_screen_size(data)
        )
        ball_start = _render_as_xy(
            data["ball_args"],
            "ball_args",
        )
        original_goal_pos = _render_as_xy(
            data["goal_args"],
            "goal_args",
        )
        goal_pos = (
            original_goal_pos[0],
            original_goal_pos[1]
            + float(goal_y_offset),
        )
        goal_x, goal_y = goal_pos

        bottom_a, bottom_b, bottom_source = (
            _render_get_json_bottom_border(data)
        )
        bottom_y_for_side = max(
            bottom_a[1],
            bottom_b[1],
        )
        side_length, side_source = (
            _render_get_json_side_border_length(
                data,
                bottom_y_for_side,
            )
        )

        scene_objects = [
            objects.Ball(
                _RenderConfig(
                    position=ball_start,
                )
            ),
            objects.Goal(
                _RenderConfig(
                    position=goal_pos,
                )
            ),
        ]

        if hasattr(objects, "LeftBorder"):
            try:
                scene_objects.append(
                    objects.LeftBorder(
                        l=side_length,
                    )
                )
            except TypeError:
                scene_objects.append(
                    objects.LeftBorder()
                )

        if hasattr(objects, "RightBorder"):
            try:
                scene_objects.append(
                    objects.RightBorder(
                        l=side_length,
                    )
                )
            except TypeError:
                scene_objects.append(
                    objects.RightBorder()
                )

        scene_objects.append(
            objects.BottomBorder(
                bottom_a,
                bottom_b,
            )
        )

        if (
            not hasattr(objects, "LeftBorder")
            and hasattr(objects, "PlinkoBorder")
        ):
            plinko_args = data.get(
                "plinko_border_args",
                [],
            )
            try:
                scene_objects.append(
                    objects.PlinkoBorder(
                        *plinko_args
                    )
                )
            except Exception as exc:
                log(
                    "Skipping PlinkoBorder because "
                    f"it failed: {exc}"
                )

        parsed_containers = []
        for idx, container in enumerate(
            data.get("container_args", [])
            or []
        ):
            try:
                parsed = _render_parse_container(
                    container,
                    idx,
                )
                parsed_containers.append(
                    parsed
                )
                scene_objects.append(
                    objects.Container(
                        _RenderConfig(
                            id=idx,
                            position=(
                                parsed["x"],
                                parsed["y"],
                            ),
                            width=parsed["width"],
                            angle=parsed["angle"],
                        )
                    )
                )
            except Exception as exc:
                log(
                    f"Skipping container {idx}: "
                    f"{container!r} because {exc}"
                )

        parsed_lines = []
        line_args, line_key = (
            _render_get_line_args_from_json(
                data
            )
        )
        for idx, line_arg in enumerate(
            line_args
        ):
            try:
                parsed = _render_parse_line(
                    line_arg,
                    idx,
                )
                parsed_lines.append(
                    parsed
                )
                if not hasattr(objects, "Line"):
                    raise RuntimeError(
                        "objects.Line does not exist "
                        "in this repo."
                    )
                scene_objects.append(
                    objects.Line(
                        _RenderConfig(
                            point_a=parsed["point_a"],
                            point_b=parsed["point_b"],
                        ),
                        angle=parsed["angle"],
                    )
                )
            except Exception as exc:
                log(
                    f"Skipping line {idx}: "
                    f"{line_arg!r} because {exc}"
                )

        if self.experiment_kind == "exp1":
            objects_list = data.get(
                "objects",
                [],
            )
            if isinstance(
                objects_list,
                list,
            ):
                has_line_object = any(
                    str(item).lower()
                    == "line"
                    for item in objects_list
                )
                if (
                    has_line_object
                    and not parsed_lines
                ):
                    log(
                        "WARNING: JSON objects includes "
                        "Line, but no line_args/lines "
                        "key was found."
                    )

        scene = Scene(
            scene_objects,
            screen_size=(
                self.RENDER_WIDTH,
                self.RENDER_HEIGHT,
            ),
        )
        scene.instantiate_scene()

        bottom_border_rule = (
            "shared_across_all_generated_exp1more_jsons"
            if self.experiment_kind == "exp1"
            else "from_json_not_glued_to_target"
        )

        static = {
            "json_file": str(json_path),
            "json_name": json_path.stem,
            "scene_name": data.get(
                "name",
                json_path.stem,
            ),
            "json_screen_width": json_width,
            "json_screen_height": json_height,
            "render_width": self.RENDER_WIDTH,
            "render_height": self.RENDER_HEIGHT,
            "fps": 60,
            "ball_start_x": ball_start[0],
            "ball_start_y": ball_start[1],
            "original_goal_x": original_goal_pos[0],
            "original_goal_y": original_goal_pos[1],
            "goal_y_offset_applied": float(
                goal_y_offset
            ),
            "goal_x": goal_x,
            "goal_y": goal_y,
            "goal_half_width": 40,
            "goal_half_height": 20,
            "goal_bottom_y": goal_y + 20,
            "left_border_x": 10.0,
            "left_border_y1": 210.0,
            "left_border_y2": float(
                side_length - 10
            ),
            "right_border_x": 790.0,
            "right_border_y1": 210.0,
            "right_border_y2": float(
                side_length - 10
            ),
            "side_border_l": float(
                side_length
            ),
            "side_border_source": side_source,
            "bottom_border_x1": bottom_a[0],
            "bottom_border_y1": bottom_a[1],
            "bottom_border_x2": bottom_b[0],
            "bottom_border_y2": bottom_b[1],
            "bottom_border_source": bottom_source,
            "bottom_border_rule": bottom_border_rule,
            "plinko_border_args_json": json.dumps(
                data.get(
                    "plinko_border_args",
                    [],
                )
            ),
            "objects_json": json.dumps(
                data.get(
                    "objects",
                    [],
                )
            ),
            "container_count": len(
                parsed_containers
            ),
            "containers_json": json.dumps(
                parsed_containers
            ),
            "line_args_key": line_key,
            "line_count": len(
                parsed_lines
            ),
            "lines_json": json.dumps(
                parsed_lines
            ),
            "full_scene_json": json.dumps(
                data
            ),
        }

        for container in parsed_containers:
            idx = container["id"]
            static[
                f"container_{idx}_x"
            ] = container["x"]
            static[
                f"container_{idx}_y"
            ] = container["y"]
            static[
                f"container_{idx}_width"
            ] = container["width"]
            static[
                f"container_{idx}_length"
            ] = container["length"]
            static[
                f"container_{idx}_angle"
            ] = container["angle"]

        for line in parsed_lines:
            idx = line["id"]
            static[
                f"line_{idx}_point_a_x"
            ] = line["point_a"][0]
            static[
                f"line_{idx}_point_a_y"
            ] = line["point_a"][1]
            static[
                f"line_{idx}_point_b_x"
            ] = line["point_b"][0]
            static[
                f"line_{idx}_point_b_y"
            ] = line["point_b"][1]
            static[
                f"line_{idx}_angle"
            ] = line["angle"]

        return (
            scene,
            self.RENDER_WIDTH,
            self.RENDER_HEIGHT,
            data,
            static,
        )

    def draw_scene(
        self,
        scene,
        width,
        height,
        frame_idx,
        json_name,
    ):
        image = Image.new(
            "RGB",
            (width, height),
            (20, 20, 24),
        )
        draw = ImageDraw.Draw(image)

        for x in range(0, width, 50):
            draw.line(
                [(x, 0), (x, height)],
                fill=(35, 35, 40),
            )

        for y in range(0, height, 50):
            draw.line(
                [(0, y), (width, y)],
                fill=(35, 35, 40),
            )

        def draw_circle(shape):
            position = shape.body.position
            radius = shape.radius
            color = getattr(
                shape,
                "color",
                None,
            )
            fill = (
                tuple(color[:3])
                if color
                else (220, 40, 40)
            )
            draw.ellipse(
                [
                    position.x - radius,
                    position.y - radius,
                    position.x + radius,
                    position.y + radius,
                ],
                fill=fill,
                outline=(30, 30, 30),
                width=2,
            )

        def draw_segment(shape):
            body = shape.body
            point_a = body.local_to_world(
                shape.a
            )
            point_b = body.local_to_world(
                shape.b
            )
            color = getattr(
                shape,
                "color",
                None,
            )
            fill = (
                tuple(color[:3])
                if color
                else (245, 245, 245)
            )
            draw.line(
                [
                    point_a.x,
                    point_a.y,
                    point_b.x,
                    point_b.y,
                ],
                fill=fill,
                width=max(
                    4,
                    int(shape.radius * 2),
                ),
            )

        def draw_poly(shape):
            body = shape.body
            vertices = [
                body.local_to_world(vertex)
                for vertex
                in shape.get_vertices()
            ]
            points = [
                (vertex.x, vertex.y)
                for vertex in vertices
            ]
            color = getattr(
                shape,
                "color",
                None,
            )
            fill = (
                tuple(color[:3])
                if color
                else (80, 200, 80)
            )
            draw.polygon(
                points,
                fill=fill,
                outline=(30, 30, 30),
            )

        for obj in scene.objects:
            for component in obj.components[1:]:
                component_name = (
                    component
                    .__class__
                    .__name__
                )

                if component_name == "Circle":
                    draw_circle(component)
                elif component_name == "Segment":
                    draw_segment(component)
                elif component_name == "Poly":
                    draw_poly(component)

        draw.text(
            (20, 20),
            (
                f"{json_name}.json | "
                f"frame {frame_idx:04d}"
            ),
            fill=(230, 230, 230),
        )

        return image


exp1_mod = _InlineRenderModule("exp1")
exp2_mod = _InlineRenderModule("exp2")

print(
    "Using inline Exp. 1 / Exp. 2 rendering logic "
    "with physics_abstraction_master."
)

In [ ]:
# ============================================================
# 5. Pymunk scene construction and simulation helpers
# ============================================================

if str(PROJECT_PYTHON) not in sys.path:
    sys.path.insert(0, str(PROJECT_PYTHON))

print("Physics project:", PROJECT_ROOT)

TMP_JSON_DIR = Path(
    tempfile.mkdtemp(
        prefix="hybrid_grid_jsons_"
    )
)

# Populated serially before multiprocessing begins.
SCENE_JSON_PATH_LOOKUP = {}

def get_render_module_for_scene(scene_dir):
    return exp1_mod if scene_kind(scene_dir) == "exp1" else exp2_mod

def get_goal_y_offset_for_scene(scene_dir):
    return -40.0 if scene_kind(scene_dir) == "exp1" else 0.0

def get_json_for_scene(scene_dir):
    """
    Prefer an actual JSON in the scene folder.
    If not available, recover full_scene_json from static_scene_settings.json
    or from simulation_dataset.csv and write a temporary JSON.
    """
    scene_dir = Path(scene_dir)

    lookup_key = str(
        scene_dir.resolve()
    )

    if (
        "SCENE_JSON_PATH_LOOKUP"
        in globals()
        and lookup_key
        in SCENE_JSON_PATH_LOOKUP
    ):
        return Path(
            SCENE_JSON_PATH_LOOKUP[
                lookup_key
            ]
        )

    jsons = sorted(scene_dir.glob("*.json"))
    jsons = [p for p in jsons if p.name not in {"metadata.json", "static_scene_settings.json"}]
    if jsons:
        return jsons[0]

    static_path = scene_dir / "static_scene_settings.json"
    if static_path.exists():
        with open(static_path, "r") as f:
            static = json.load(f)
        if "full_scene_json" in static:
            data = json.loads(static["full_scene_json"])
            out = TMP_JSON_DIR / f"{safe_scene_id(scene_dir)}.json"
            with open(out, "w") as f:
                json.dump(data, f, indent=2)
            return out

    csv_path = scene_dir / "simulation_dataset.csv"
    if csv_path.exists():
        df0 = pd.read_csv(csv_path, nrows=1)
        if "full_scene_json" in df0.columns:
            data = json.loads(df0.iloc[0]["full_scene_json"])
            out = TMP_JSON_DIR / f"{safe_scene_id(scene_dir)}.json"
            with open(out, "w") as f:
                json.dump(data, f, indent=2)
            return out

    raise FileNotFoundError(
        f"Could not find/recover scene JSON for {scene_dir}. "
        "Need either a .json file, static_scene_settings.json with full_scene_json, "
        "or simulation_dataset.csv with full_scene_json."
    )

def silent_log(msg):
    pass

def find_ball_object(scene):
    for obj in scene.objects:
        if obj.name == "Ball":
            return obj
    raise RuntimeError("No Ball object found.")

def force_ball_render_style(scene):
    """
    Make the simulated ball render like the original training/test ball.

    This is important for RNN feedback: simulated feedback frames should look
    like normal scene frames, not like diagnostic overlay plots. In particular,
    the simulated ball should be drawn as the same red ball used in the original
    rendered frames, not as a green simulation marker.
    """
    for obj in scene.objects:
        if getattr(obj, "name", "") == "Ball":
            for component in obj.components[1:]:
                # draw_scene uses component.color when available.
                try:
                    component.color = (220, 40, 40)
                except Exception:
                    pass

def build_pymunk_scene(scene_dir, start_position_xy, start_velocity_xy):
    mod = get_render_module_for_scene(scene_dir)
    json_path = get_json_for_scene(scene_dir)
    goal_y_offset = get_goal_y_offset_for_scene(scene_dir)

    scene, width, height, raw_data, static = mod.build_scene_from_json(
        json_path,
        silent_log,
        goal_y_offset,
    )
    ball_obj = find_ball_object(scene)
    ball_obj.body.position = (float(start_position_xy[0]), float(start_position_xy[1]))
    ball_obj.body.velocity = (float(start_velocity_xy[0]), float(start_velocity_xy[1]))

    try:
        for obj in scene.objects:
            scene.physics.space.reindex_static()
            scene.physics.space.reindex_shapes_for_body(obj.body)
    except Exception:
        pass

    return scene, ball_obj, mod, width, height, json_path.stem

def render_scene_to_cache_array(mod, scene, width, height, frame_idx, json_name):
    """
    Render a simulated scene into the exact visual format used for RNN feedback.

    This renders the full physical scene with the Ball object styled as the
    normal red ball from the training videos. It intentionally does not draw
    green simulation dots, coral abstraction dots, or any diagnostic overlay
    graphics onto the feedback image.
    """
    force_ball_render_style(scene)
    img = mod.draw_scene(scene, width, height, frame_idx, json_name).convert("RGB")
    img_small = img.resize((IMAGE_W, IMAGE_H), Image.BILINEAR)
    arr = np.asarray(img_small, dtype=np.uint8)
    arr = np.transpose(arr, (2, 0, 1))  # 3 x H x W
    return arr, img

def simulate_next_frames(
    scene_dir,
    start_frame_idx,
    start_position_xy,
    start_velocity_xy,
    terminal_geometry,
    n_frames=SIM_CHUNK_FRAMES,
):
    """
    CPU Pymunk rollout from a given position and velocity.

    start_position_xy is the current-frame position.
    Returned positions correspond to frames
    start_frame_idx + 1, ..., start_frame_idx + n_frames.

    Scene ending uses the same geometry rule as abstraction:
      - goal rectangle distance <= 20 px;
      - finite ground-segment distance <= 40 px;
      - goal checked first.

    Actual Pymunk collision flags are not used to determine the
    terminal event.
    """
    scene, ball_obj, mod, width, height, json_name = build_pymunk_scene(
        scene_dir,
        start_position_xy=start_position_xy,
        start_velocity_xy=start_velocity_xy,
    )

    rows = []
    cache_frames = []
    full_images = []
    stop_reason = ""

    for step in range(1, int(n_frames) + 1):
        try:
            for obj in scene.objects:
                scene.physics.space.reindex_static()
                scene.physics.space.reindex_shapes_for_body(obj.body)
        except Exception:
            pass

        scene.physics.forward()

        pos = ball_obj.body.position
        vel = ball_obj.body.velocity
        frame_idx = int(start_frame_idx + step)

        terminal_event = terminal_event_at_point(
            x=float(pos.x),
            y=float(pos.y),
            geometry=terminal_geometry,
        )

        arr, img = render_scene_to_cache_array(
            mod,
            scene,
            width,
            height,
            frame_idx,
            json_name,
        )
        cache_frames.append(arr)
        full_images.append(img)

        rows.append({
            "frame": frame_idx,
            "x": float(pos.x),
            "y": float(pos.y),
            "vx": float(vel.x),
            "vy": float(vel.y),
            "step": step,
            "simulation_terminal_event": terminal_event,
        })

        if terminal_event == "goal":
            stop_reason = "goal_distance_threshold"
            break

        if terminal_event == "ground":
            stop_reason = "ground_distance_threshold"
            break

    return (
        pd.DataFrame(rows),
        np.asarray(cache_frames, dtype=np.uint8),
        full_images,
        stop_reason,
    )


In [ ]:
# ============================================================
# 6. RNN prediction and hybrid rollout
# ============================================================

def pil_images_to_rnn_history_array(frame_images):
    """
    Convert exactly N_HISTORY rendered PIL images into the same cached-frame
    format used during training:

      PIL RGB image
      -> resize to (IMAGE_W, IMAGE_H) with Image.BILINEAR
      -> uint8 H x W x 3
      -> transpose to 3 x H x W
      -> stack into N_HISTORY x 3 x IMAGE_H x IMAGE_W

    The RNN inference function then converts this uint8 array to float32 / 255.0
    and adds the batch dimension: 1 x N_HISTORY x 3 x IMAGE_H x IMAGE_W.
    """
    if len(frame_images) != N_HISTORY:
        raise RuntimeError(
            f"Expected exactly {N_HISTORY} rendered feedback frames, got {len(frame_images)}."
        )

    arrs = []
    for img in frame_images:
        img = img.convert("RGB")
        img_small = img.resize((IMAGE_W, IMAGE_H), Image.BILINEAR)
        arr = np.asarray(img_small, dtype=np.uint8)      # H x W x 3
        arr = np.transpose(arr, (2, 0, 1))               # 3 x H x W
        arrs.append(arr)

    history = np.stack(arrs, axis=0).astype(np.uint8)    # T x 3 x H x W

    expected_shape = (N_HISTORY, 3, IMAGE_H, IMAGE_W)
    if history.shape != expected_shape:
        raise RuntimeError(f"Bad feedback history shape: {history.shape}; expected {expected_shape}")
    if history.dtype != np.uint8:
        raise RuntimeError(f"Bad feedback history dtype: {history.dtype}; expected uint8")

    return history


def validate_rnn_history_array(history_frames_uint8, context="history"):
    expected_shape = (N_HISTORY, 3, IMAGE_H, IMAGE_W)
    if not isinstance(history_frames_uint8, np.ndarray):
        raise RuntimeError(f"{context} must be a numpy array, got {type(history_frames_uint8)}")
    if history_frames_uint8.shape != expected_shape:
        raise RuntimeError(f"{context} has shape {history_frames_uint8.shape}; expected {expected_shape}")
    if history_frames_uint8.dtype != np.uint8:
        raise RuntimeError(f"{context} has dtype {history_frames_uint8.dtype}; expected uint8")


def predict_future_from_images(history_frames_uint8, remaining_future_frames):
    """
    history_frames_uint8: N_HISTORY x 3 x IMAGE_H x IMAGE_W, uint8.

    This is exactly the same representation as the compressed frame caches.
    The model input is converted here to:
      1 x N_HISTORY x 3 x IMAGE_H x IMAGE_W, float32 in [0, 1].

    Returns a DataFrame with offset, pred_x, pred_y, predicted_error_pixels.
    """
    validate_rnn_history_array(history_frames_uint8, context="RNN input history")

    M = int(remaining_future_frames)
    if M <= 0:
        return pd.DataFrame(columns=["offset", "pred_x", "pred_y", "predicted_error_pixels"])

    offsets = np.arange(1, M + 1, dtype=np.float32)
    checkpoint_times = offsets / float(GLOBAL_MAX_FUTURE_OFFSET)

    X_np = history_frames_uint8.astype(np.float32) / 255.0
    X = torch.from_numpy(X_np).unsqueeze(0).to(DEVICE)              # 1 x T x 3 x H x W
    times = torch.from_numpy(checkpoint_times).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        with autocast_context():
            pred_pos_norm, pred_err_raw = model(X, times)

    pred_pos_norm_np = pred_pos_norm.float().cpu().numpy()[0]
    pred_pos_pix = unnormalize_y(pred_pos_norm_np)

    pred_err_norm = positive_error_magnitude(pred_err_raw).float().cpu().numpy()[0]
    pred_err_pix = pred_err_norm * ERR_MAG_PIXEL_SCALE_FOR_PLOTS

    return pd.DataFrame({
        "offset": offsets.astype(int),
        "pred_x": pred_pos_pix[:, 0].astype(float),
        "pred_y": pred_pos_pix[:, 1].astype(float),
        "predicted_error_pixels": pred_err_pix.astype(float),
    })


def estimate_velocity_for_sim(history_positions, accepted_pred_positions):
    """
    Fallback velocity estimate for cases with no accepted abstraction step.
    The RNN does not explicitly predict velocity, so use the last observed
    finite difference and convert pixels/frame to Pymunk velocity units.
    """
    if len(accepted_pred_positions) >= 2:
        v_pixels_per_frame = np.asarray(accepted_pred_positions[-1]) - np.asarray(accepted_pred_positions[-2])
    elif len(accepted_pred_positions) == 1:
        v_pixels_per_frame = np.asarray(accepted_pred_positions[-1]) - np.asarray(history_positions[-1])
    else:
        v_pixels_per_frame = np.asarray(history_positions[-1]) - np.asarray(history_positions[-2])

    return v_pixels_per_frame * FPS_FOR_VELOCITY


def estimate_straight_line_velocity_for_sim(start_position, endpoint_position, n_steps):
    """
    Velocity for the new straight-line abstraction rule.

    If abstraction accepts k steps, the hybrid abstraction is treated as one
    straight line from the current position to the accepted endpoint. The
    simulation starts with direction equal to that straight line and speed equal
    to line_length / k, which is equivalent to displacement / k in pixels/frame.
    Convert pixels/frame to Pymunk velocity units using FPS_FOR_VELOCITY.
    """
    start_position = np.asarray(start_position, dtype=float)
    endpoint_position = np.asarray(endpoint_position, dtype=float)
    n_steps = max(int(n_steps), 1)
    v_pixels_per_frame = (endpoint_position - start_position) / float(n_steps)
    return v_pixels_per_frame * FPS_FOR_VELOCITY



def chunk_has_error_above_threshold(
    pred_df,
    threshold_pixels,
    offset_start,
    offset_end,
):
    """
    Check only the full-span endpoint error.

    The endpoint is `offset_end`; intermediate RNN prediction errors are
    intentionally ignored. The function keeps the prior return signature for
    compatibility: (failed, bad_offset, checked_endpoint_df).
    """
    endpoint_df = pred_df.loc[
        pred_df["offset"].astype(int).eq(
            int(offset_end)
        )
    ].tail(1).copy()

    if len(endpoint_df) == 0:
        return True, int(offset_end), endpoint_df

    endpoint_error = pd.to_numeric(
        endpoint_df[
            "predicted_error_pixels"
        ],
        errors="coerce",
    ).iloc[0]

    failed = (
        not np.isfinite(
            float(endpoint_error)
        )
        or float(endpoint_error)
        > float(threshold_pixels)
    )

    return (
        bool(failed),
        int(offset_end) if failed else None,
        endpoint_df,
    )



def predict_future_offsets_from_images(
    history_frames_uint8,
    offset_start,
    offset_end,
):
    """
    Predict only the requested offsets from one fixed RNN history.

    Endpoint-only abstraction checks call this with
    `offset_start == offset_end`, so no intermediate RNN positions or errors
    are requested for the decision.
    """
    validate_rnn_history_array(
        history_frames_uint8,
        context="RNN input history",
    )

    offset_start = int(
        offset_start
    )
    offset_end = int(
        offset_end
    )

    if offset_end < offset_start:
        return pd.DataFrame(
            columns=[
                "offset",
                "pred_x",
                "pred_y",
                "predicted_error_pixels",
            ]
        )

    offsets = np.arange(
        offset_start,
        offset_end + 1,
        dtype=np.float32,
    )

    checkpoint_times = (
        offsets
        / float(
            GLOBAL_MAX_FUTURE_OFFSET
        )
    )

    X_np = (
        history_frames_uint8.astype(
            np.float32
        )
        / 255.0
    )

    X = (
        torch.from_numpy(
            X_np
        )
        .unsqueeze(
            0
        )
        .to(
            DEVICE
        )
    )

    times = (
        torch.from_numpy(
            checkpoint_times
        )
        .unsqueeze(
            0
        )
        .to(
            DEVICE
        )
    )

    with torch.no_grad():
        with autocast_context():
            pred_pos_norm, pred_err_raw = model(
                X,
                times,
            )

    pred_pos_norm_np = (
        pred_pos_norm
        .float()
        .cpu()
        .numpy()[0]
    )

    pred_pos_pix = unnormalize_y(
        pred_pos_norm_np
    )

    pred_err_norm = (
        positive_error_magnitude(
            pred_err_raw
        )
        .float()
        .cpu()
        .numpy()[0]
    )

    pred_err_pix = (
        pred_err_norm
        * ERR_MAG_PIXEL_SCALE_FOR_PLOTS
    )

    return pd.DataFrame(
        {
            "offset": offsets.astype(
                int
            ),
            "pred_x": pred_pos_pix[
                :,
                0,
            ].astype(
                float
            ),
            "pred_y": pred_pos_pix[
                :,
                1,
            ].astype(
                float
            ),
            "predicted_error_pixels": (
                pred_err_pix.astype(
                    float
                )
            ),
        }
    )


def _unwrap_singletons(value):
    while isinstance(value, (list, tuple)) and len(value) == 1:
        value = value[0]
    return value


def _parse_xy(value):
    value = _unwrap_singletons(value)
    if isinstance(value, dict):
        for key in ("position", "pos", "point", "location"):
            if key in value:
                value = _unwrap_singletons(value[key])
                break
    if isinstance(value, (list, tuple)) and len(value) >= 2:
        return float(value[0]), float(value[1])
    raise ValueError(f"Could not parse x/y from {value!r}")


def _finite_value(row, names, default=np.nan):
    for name in names:
        if name in row.index:
            value = pd.to_numeric(pd.Series([row[name]]), errors="coerce").iloc[0]
            if pd.notna(value) and np.isfinite(float(value)):
                return float(value)
    return float(default)


def load_terminal_geometry(scene_dir):
    """
    Load only the static geometry needed to decide whether an RNN-predicted
    path has reached the goal or the bottom border.

    The true number of frames is never loaded or used for stopping.
    """
    scene_dir = Path(scene_dir)
    csv_path = scene_dir / "simulation_dataset.csv"

    first_row = pd.Series(dtype=object)
    if csv_path.exists():
        try:
            first_row = pd.read_csv(csv_path, nrows=1).iloc[0]
        except Exception:
            first_row = pd.Series(dtype=object)

    ball_radius = _finite_value(first_row, ["ball_radius"], default=20.0)
    bottom_border_radius = _finite_value(
        first_row,
        ["bottom_border_radius", "ground_radius"],
        default=10.0,
    )

    goal_x = _finite_value(first_row, ["goal_x"])
    goal_y = _finite_value(first_row, ["goal_y"])
    goal_half_width = _finite_value(first_row, ["goal_half_width"], default=40.0)
    goal_half_height = _finite_value(first_row, ["goal_half_height"], default=20.0)

    goal_left = _finite_value(first_row, ["goal_left"], default=goal_x - goal_half_width)
    goal_right = _finite_value(first_row, ["goal_right"], default=goal_x + goal_half_width)
    goal_top = _finite_value(first_row, ["goal_top"], default=goal_y - goal_half_height)
    goal_bottom = _finite_value(first_row, ["goal_bottom"], default=goal_y + goal_half_height)

    bx1 = _finite_value(first_row, ["bottom_border_x1"])
    by1 = _finite_value(first_row, ["bottom_border_y1"])
    bx2 = _finite_value(first_row, ["bottom_border_x2"])
    by2 = _finite_value(first_row, ["bottom_border_y2"])

    need_json = not all(np.isfinite(v) for v in [
        goal_x, goal_y, goal_left, goal_right, goal_top, goal_bottom,
        bx1, by1, bx2, by2,
    ])

    if need_json:
        json_path = get_json_for_scene(scene_dir)
        with open(json_path, "r") as f:
            data = json.load(f)

        if not np.isfinite(goal_x) or not np.isfinite(goal_y):
            goal_x_json, goal_y_json = _parse_xy(data.get("goal_args"))
            goal_x = float(goal_x_json)
            goal_y = float(goal_y_json + get_goal_y_offset_for_scene(scene_dir))

        if not np.isfinite(goal_left):
            goal_left = goal_x - goal_half_width
        if not np.isfinite(goal_right):
            goal_right = goal_x + goal_half_width
        if not np.isfinite(goal_top):
            goal_top = goal_y - goal_half_height
        if not np.isfinite(goal_bottom):
            goal_bottom = goal_y + goal_half_height

        if not all(np.isfinite(v) for v in [bx1, by1, bx2, by2]):
            bottom_args = _unwrap_singletons(data.get("bottom_border_args"))
            if isinstance(bottom_args, (list, tuple)) and len(bottom_args) >= 2:
                bx1, by1 = _parse_xy(bottom_args[0])
                bx2, by2 = _parse_xy(bottom_args[1])
            else:
                raise ValueError(
                    f"Could not parse bottom_border_args for {scene_dir}: {bottom_args!r}"
                )

    geometry = {
        "goal_center_x": float(goal_x),
        "goal_center_y": float(goal_y),
        "goal_half_width": float(goal_half_width),
        "goal_half_height": float(goal_half_height),
        "bottom_x1": float(bx1),
        "bottom_y1": float(by1),
        "bottom_x2": float(bx2),
        "bottom_y2": float(by2),
    }

    if not all(np.isfinite(v) for v in geometry.values()):
        raise ValueError(f"Non-finite terminal geometry for {scene_dir}: {geometry}")

    return geometry


def point_to_segment_distance(
    point,
    segment_start,
    segment_end,
):
    """
    Euclidean distance from a point to a finite line segment.
    """
    point = np.asarray(
        point,
        dtype=np.float64,
    )
    segment_start = np.asarray(
        segment_start,
        dtype=np.float64,
    )
    segment_end = np.asarray(
        segment_end,
        dtype=np.float64,
    )

    segment_vector = (
        segment_end
        - segment_start
    )

    segment_length_squared = float(
        np.dot(
            segment_vector,
            segment_vector,
        )
    )

    if segment_length_squared <= 0:
        return float(
            np.linalg.norm(
                point
                - segment_start
            )
        )

    projection = float(
        np.dot(
            point - segment_start,
            segment_vector,
        )
        / segment_length_squared
    )

    projection = float(
        np.clip(
            projection,
            0.0,
            1.0,
        )
    )

    closest_point = (
        segment_start
        + projection
        * segment_vector
    )

    return float(
        np.linalg.norm(
            point
            - closest_point
        )
    )


def distance_to_axis_aligned_rectangle(
    point,
    rectangle_center,
    half_width,
    half_height,
):
    """
    Distance from a point to an axis-aligned rectangle.

    Returns zero when the point lies inside the rectangle.
    """
    point = np.asarray(
        point,
        dtype=np.float64,
    )

    rectangle_center = np.asarray(
        rectangle_center,
        dtype=np.float64,
    )

    delta = np.abs(
        point
        - rectangle_center
    )

    outside_x = max(
        float(
            delta[0]
            - half_width
        ),
        0.0,
    )

    outside_y = max(
        float(
            delta[1]
            - half_height
        ),
        0.0,
    )

    return float(
        np.hypot(
            outside_x,
            outside_y,
        )
    )


def terminal_event_at_point(
    x,
    y,
    geometry,
):
    """
    Use a 20-pixel goal threshold and a 40-pixel ground threshold.

    Goal is checked first when both terminal regions are reached.
    """
    point = np.array(
        [
            float(x),
            float(y),
        ],
        dtype=np.float64,
    )

    goal_distance = (
        distance_to_axis_aligned_rectangle(
            point=point,
            rectangle_center=np.array(
                [
                    geometry[
                        "goal_center_x"
                    ],
                    geometry[
                        "goal_center_y"
                    ],
                ],
                dtype=np.float64,
            ),
            half_width=geometry[
                "goal_half_width"
            ],
            half_height=geometry[
                "goal_half_height"
            ],
        )
    )

    if (
        goal_distance
        <= GOAL_CONTACT_DISTANCE_PX
    ):
        return "goal"

    ground_distance = (
        point_to_segment_distance(
            point=point,
            segment_start=np.array(
                [
                    geometry[
                        "bottom_x1"
                    ],
                    geometry[
                        "bottom_y1"
                    ],
                ],
                dtype=np.float64,
            ),
            segment_end=np.array(
                [
                    geometry[
                        "bottom_x2"
                    ],
                    geometry[
                        "bottom_y2"
                    ],
                ],
                dtype=np.float64,
            ),
        )
    )

    if (
        ground_distance
        <= GROUND_CONTACT_DISTANCE_PX
    ):
        return "ground"

    return ""



def detect_terminal_event_on_straight_line(
    start_position,
    endpoint_position,
    n_steps,
    geometry,
):
    """
    Find the first terminal point on one accepted straight-line abstraction.

    The line is sampled at its integer frame steps. Every sampled point is
    obtained by linear interpolation between the current position and the
    full-span RNN endpoint. No intermediate RNN prediction point is used.
    """
    start_position = np.asarray(
        start_position,
        dtype=np.float64,
    )

    endpoint_position = np.asarray(
        endpoint_position,
        dtype=np.float64,
    )

    if (
        start_position.shape
        != (
            2,
        )
        or endpoint_position.shape
        != (
            2,
        )
        or not np.isfinite(
            start_position
        ).all()
        or not np.isfinite(
            endpoint_position
        ).all()
    ):
        raise ValueError(
            "Straight-line terminal detection requires "
            "finite two-dimensional endpoints."
        )

    n_steps = max(
        int(
            n_steps
        ),
        1,
    )

    displacement = (
        endpoint_position
        - start_position
    )

    for step_within_span in range(
        1,
        n_steps + 1,
    ):
        fraction = (
            float(
                step_within_span
            )
            / float(
                n_steps
            )
        )

        point = (
            start_position
            + fraction
            * displacement
        )

        event = terminal_event_at_point(
            point[
                0
            ],
            point[
                1
            ],
            geometry,
        )

        if event:
            return {
                "terminal_event": str(
                    event
                ),
                "terminal_step_within_span": int(
                    step_within_span
                ),
                "terminal_fraction": float(
                    fraction
                ),
                "terminal_point": point.astype(
                    float
                ),
            }

    return {
        "terminal_event": "",
        "terminal_step_within_span": None,
        "terminal_fraction": np.nan,
        "terminal_point": None,
    }



def check_next_fixed_abstraction_candidate(
    history_frames,
    start_position,
    terminal_geometry,
    threshold_pixels,
    offset_cursor,
):
    """
    Test exactly one full A-frame abstraction span.

    Only the cumulative full-span endpoint is requested from the RNN. The
    candidate passes when that endpoint's predicted error is finite and
    `<= threshold_pixels`. Intermediate RNN positions and errors are never
    consulted.

    Terminal contact is evaluated separately along the straight line from the
    current position to the full-span endpoint. This line check does not shorten
    the endpoint used for the error decision.
    """
    offset_start = int(
        offset_cursor
        + 1
    )

    offset_end_requested = int(
        offset_cursor
        + ABSTRACTION_CHECK_CHUNK_FRAMES
    )

    pred_df = predict_future_offsets_from_images(
        history_frames_uint8=history_frames,
        offset_start=offset_end_requested,
        offset_end=offset_end_requested,
    )

    chunk_failed, first_bad_offset, endpoint_df = (
        chunk_has_error_above_threshold(
            pred_df=pred_df,
            threshold_pixels=threshold_pixels,
            offset_start=offset_start,
            offset_end=offset_end_requested,
        )
    )

    if len(
        endpoint_df
    ) == 0:
        return {
            "can_check": False,
            "chunk_failed": True,
            "chunk_len": int(
                ABSTRACTION_CHECK_CHUNK_FRAMES
            ),
            "offset_start": int(
                offset_start
            ),
            "offset_end_requested": int(
                offset_end_requested
            ),
            "offset_end_effective": None,
            "first_bad_offset": int(
                offset_end_requested
            ),
            "checked_df": endpoint_df,
            "pred_df": pred_df,
            "endpoint_row": None,
            "endpoint_error_pixels": np.nan,
            "requested_endpoint_position": None,
            "terminal_event": "",
            "terminal_step_within_span": None,
            "terminal_fraction": np.nan,
            "terminal_point": None,
            "reason": "no_full_span_endpoint_prediction",
        }

    endpoint_row = endpoint_df.iloc[
        -1
    ]

    endpoint_error = pd.to_numeric(
        pd.Series(
            [
                endpoint_row[
                    "predicted_error_pixels"
                ]
            ]
        ),
        errors="coerce",
    ).iloc[
        0
    ]

    requested_endpoint_position = np.array(
        [
            float(
                endpoint_row[
                    "pred_x"
                ]
            ),
            float(
                endpoint_row[
                    "pred_y"
                ]
            ),
        ],
        dtype=float,
    )

    line_terminal = (
        detect_terminal_event_on_straight_line(
            start_position=start_position,
            endpoint_position=requested_endpoint_position,
            n_steps=ABSTRACTION_CHECK_CHUNK_FRAMES,
            geometry=terminal_geometry,
        )
    )

    if not np.isfinite(
        float(
            endpoint_error
        )
    ):
        reason = (
            "full_span_endpoint_error_nonfinite"
        )
    elif chunk_failed:
        reason = (
            "full_span_endpoint_error_above_threshold"
        )
    else:
        reason = (
            "full_span_endpoint_error_below_threshold"
        )

    return {
        "can_check": True,
        "chunk_failed": bool(
            chunk_failed
        ),
        "chunk_len": int(
            ABSTRACTION_CHECK_CHUNK_FRAMES
        ),
        "offset_start": int(
            offset_start
        ),
        "offset_end_requested": int(
            offset_end_requested
        ),
        "offset_end_effective": int(
            offset_end_requested
        ),
        "first_bad_offset": (
            int(
                first_bad_offset
            )
            if first_bad_offset
            is not None
            else None
        ),
        "checked_df": endpoint_df,
        "pred_df": pred_df,
        "endpoint_row": endpoint_row,
        "endpoint_error_pixels": float(
            endpoint_error
        ),
        "requested_endpoint_position": (
            requested_endpoint_position
        ),
        "terminal_event": str(
            line_terminal[
                "terminal_event"
            ]
        ),
        "terminal_step_within_span": (
            line_terminal[
                "terminal_step_within_span"
            ]
        ),
        "terminal_fraction": float(
            line_terminal[
                "terminal_fraction"
            ]
        ),
        "terminal_point": (
            line_terminal[
                "terminal_point"
            ]
        ),
        "reason": reason,
    }



def run_adaptive_simulation_segment(
    scene_dir,
    threshold_pixels,
    sim_start_frame,
    sim_start_position,
    sim_start_velocity,
    terminal_geometry,
):
    """
    Adaptive Pymunk simulation without access to the true scene length.

    Starting from the same state, try S, 2S, 3S, ... simulated frames.
    A candidate is accepted when:
      - the simulated ball enters the goal-20/ground-40 terminal region; or
      - the full endpoint error of the next fixed A-frame abstraction is
        finite and below threshold.

    ADAPTIVE_SIM_MAX_FRAMES is only a safety cap and is unrelated to the
    ground-truth number of frames.
    """
    max_frames = int(
        ADAPTIVE_SIM_MAX_FRAMES
    )

    candidate_n = min(
        int(
            SIM_CHUNK_FRAMES
        ),
        max_frames,
    )

    attempt = 0
    last_result = None

    while True:
        attempt += 1

        (
            sim_df,
            sim_cache_frames,
            sim_full_images,
            stop_reason,
        ) = simulate_next_frames(
            scene_dir=scene_dir,
            start_frame_idx=sim_start_frame,
            start_position_xy=sim_start_position,
            start_velocity_xy=sim_start_velocity,
            terminal_geometry=terminal_geometry,
            n_frames=candidate_n,
        )

        accepted_frames = int(
            len(
                sim_df
            )
        )

        accept_reason = None
        next_check_error = np.nan
        next_check_terminal_event = ""

        if accepted_frames == 0:
            accept_reason = (
                "simulation_returned_no_frames"
            )

        elif stop_reason:
            accept_reason = (
                f"terminal_{stop_reason}"
            )

        elif (
            accepted_frames
            < N_HISTORY
            or len(
                sim_full_images
            )
            < N_HISTORY
        ):
            accept_reason = (
                "fewer_than_history_feedback_frames"
            )

        else:
            preview_history = (
                pil_images_to_rnn_history_array(
                    sim_full_images[
                        -N_HISTORY:
                    ]
                )
            )

            validate_rnn_history_array(
                preview_history,
                context=(
                    "adaptive simulation preview history"
                ),
            )

            preview_start_position = (
                sim_df.iloc[
                    -1
                ][
                    [
                        "x",
                        "y",
                    ]
                ]
                .to_numpy(
                    dtype=float
                )
            )

            preview_check = (
                check_next_fixed_abstraction_candidate(
                    history_frames=preview_history,
                    start_position=preview_start_position,
                    terminal_geometry=terminal_geometry,
                    threshold_pixels=threshold_pixels,
                    offset_cursor=0,
                )
            )

            next_check_terminal_event = str(
                preview_check.get(
                    "terminal_event",
                    "",
                )
            )

            preview_endpoint_error = (
                preview_check.get(
                    "endpoint_error_pixels",
                    np.nan,
                )
            )

            if np.isfinite(
                preview_endpoint_error
            ):
                next_check_error = float(
                    preview_endpoint_error
                )

            if (
                preview_check[
                    "can_check"
                ]
                and not preview_check[
                    "chunk_failed"
                ]
            ):
                accept_reason = (
                    "next_abstraction_full_endpoint_below_threshold"
                )
            else:
                accept_reason = None

        last_result = (
            sim_df,
            sim_cache_frames,
            sim_full_images,
            stop_reason,
            {
                "adaptive_sim_requested_frames": int(
                    candidate_n
                ),
                "adaptive_sim_accepted_frames": int(
                    accepted_frames
                ),
                "adaptive_sim_num_attempts": int(
                    attempt
                ),
                "adaptive_sim_next_check_error_pixels": (
                    float(
                        next_check_error
                    )
                    if np.isfinite(
                        next_check_error
                    )
                    else np.nan
                ),
                "adaptive_sim_next_check_terminal_event": (
                    next_check_terminal_event
                ),
                "adaptive_sim_accept_reason": (
                    accept_reason
                    if accept_reason
                    is not None
                    else (
                        "next_abstraction_full_endpoint_above_threshold_try_longer"
                    )
                ),
            },
        )

        if accept_reason is not None:
            return last_result

        next_candidate_n = min(
            candidate_n
            + int(
                ADAPTIVE_SIM_INCREMENT_FRAMES
            ),
            max_frames,
        )

        if next_candidate_n <= candidate_n:
            (
                sim_df,
                sim_cache_frames,
                sim_full_images,
                stop_reason,
                meta,
            ) = last_result

            meta[
                "adaptive_sim_accept_reason"
            ] = (
                "safety_max_sim_length_reached_with_"
                "next_abstraction_full_endpoint_above_threshold"
            )

            return (
                sim_df,
                sim_cache_frames,
                sim_full_images,
                stop_reason,
                meta,
            )

        candidate_n = (
            next_candidate_n
        )



def run_hybrid_for_scene(
    scene_dir,
    threshold_pixels,
    max_segments=200,
):
    """
    Time-only hybrid rollout using the current global A and S values.

    The algorithm never reads or uses the true total frame count for stopping.

    From each fixed 15-frame RNN history:
      1. Query only the full endpoint of the next A-frame abstraction span.
      2. Accept the span only if that endpoint's predicted error is finite
         and <= e.
      3. Represent the accepted span as one straight line from the current
         position to the full-span endpoint.
      4. Check goal/ground contact on linearly interpolated points along that
         straight line, never on intermediate RNN predictions.
      5. If the line reaches a terminal condition early, store its first
         terminal point as the abstraction endpoint while retaining the full
         endpoint error as the acceptance evidence.
      6. If the full endpoint fails threshold, switch to adaptive Pymunk
         simulation for S, 2S, 3S, ... frames from the same current state.

    MAX_TIME_ONLY_ROLLOUT_FRAMES and max_segments are safeguards only.
    """
    scene_dir = Path(
        scene_dir
    )

    pos_df = load_scene_positions(
        scene_dir
    )

    frames_cache = (
        load_scene_frames_cache(
            scene_dir
        )
    )

    terminal_geometry = (
        load_terminal_geometry(
            scene_dir
        )
    )

    if len(
        frames_cache
    ) < N_HISTORY:
        raise RuntimeError(
            "Not enough cached frames for "
            f"{scene_dir}: {len(frames_cache)}"
        )

    history_frames = np.asarray(
        frames_cache[
            :N_HISTORY
        ],
        dtype=np.uint8,
    )

    validate_rnn_history_array(
        history_frames,
        context=(
            "initial true cached history"
        ),
    )

    history_positions = (
        pos_df.iloc[
            :N_HISTORY
        ][
            [
                "ball_x",
                "ball_y",
            ]
        ]
        .to_numpy(
            dtype=float
        )
        .tolist()
    )

    current_frame = (
        N_HISTORY
        - 1
    )

    current_position = np.asarray(
        history_positions[
            -1
        ],
        dtype=float,
    )

    hybrid_rows = []
    segment_id = 0
    rollout_terminal_event = ""
    rollout_stop_reason = ""

    while (
        not rollout_terminal_event
        and current_frame
        < int(
            MAX_TIME_ONLY_ROLLOUT_FRAMES
        )
        and segment_id
        < int(
            max_segments
        )
    ):
        rnn_start_frame = int(
            current_frame
        )

        offset_cursor = 0
        first_bad_offset = None
        last_straight_line_velocity = None
        candidate_failed = False

        while (
            not rollout_terminal_event
            and current_frame
            < int(
                MAX_TIME_ONLY_ROLLOUT_FRAMES
            )
            and segment_id
            < int(
                max_segments
            )
        ):
            candidate = (
                check_next_fixed_abstraction_candidate(
                    history_frames=history_frames,
                    start_position=current_position,
                    terminal_geometry=terminal_geometry,
                    threshold_pixels=threshold_pixels,
                    offset_cursor=offset_cursor,
                )
            )

            if not candidate[
                "can_check"
            ]:
                candidate_failed = True
                rollout_stop_reason = (
                    candidate[
                        "reason"
                    ]
                )
                break

            first_bad_offset = (
                candidate[
                    "first_bad_offset"
                ]
            )

            if candidate[
                "chunk_failed"
            ]:
                candidate_failed = True
                break

            offset_start = int(
                candidate[
                    "offset_start"
                ]
            )

            offset_end_requested = int(
                candidate[
                    "offset_end_requested"
                ]
            )

            accepted_len = int(
                candidate[
                    "chunk_len"
                ]
            )

            endpoint_error = float(
                candidate[
                    "endpoint_error_pixels"
                ]
            )

            requested_endpoint_xy = np.asarray(
                candidate[
                    "requested_endpoint_position"
                ],
                dtype=float,
            )

            requested_endpoint_frame = int(
                rnn_start_frame
                + offset_end_requested
            )

            line_start_frame = int(
                current_frame
            )

            line_start_position = np.asarray(
                current_position,
                dtype=float,
            )

            terminal_event = str(
                candidate[
                    "terminal_event"
                ]
            )

            terminal_step_within_span = (
                candidate[
                    "terminal_step_within_span"
                ]
            )

            terminal_fraction = float(
                candidate[
                    "terminal_fraction"
                ]
            )

            if terminal_event:
                if (
                    terminal_step_within_span
                    is None
                    or candidate[
                        "terminal_point"
                    ]
                    is None
                ):
                    raise RuntimeError(
                        "A terminal abstraction candidate "
                        "is missing its straight-line contact point."
                    )

                actual_step_within_span = int(
                    terminal_step_within_span
                )

                endpoint_frame = int(
                    line_start_frame
                    + actual_step_within_span
                )

                endpoint_xy = np.asarray(
                    candidate[
                        "terminal_point"
                    ],
                    dtype=float,
                )

                offset_end_stored = int(
                    offset_cursor
                    + actual_step_within_span
                )

            else:
                actual_step_within_span = int(
                    accepted_len
                )

                endpoint_frame = int(
                    requested_endpoint_frame
                )

                endpoint_xy = (
                    requested_endpoint_xy.copy()
                )

                offset_end_stored = int(
                    offset_end_requested
                )

            straight_line_velocity = (
                estimate_straight_line_velocity_for_sim(
                    start_position=line_start_position,
                    endpoint_position=requested_endpoint_xy,
                    n_steps=accepted_len,
                )
            )

            last_straight_line_velocity = (
                straight_line_velocity
            )

            common = {
                "scene": str(
                    scene_dir
                ),
                "threshold_pixels": float(
                    threshold_pixels
                ),
                "segment_id": int(
                    segment_id
                ),
                "source": "abstraction",
                "requested_abstraction_span": int(
                    ABSTRACTION_CHECK_CHUNK_FRAMES
                ),
                "abstraction_steps": int(
                    accepted_len
                ),
                "abstraction_executed_line_steps": int(
                    actual_step_within_span
                ),
                "abstraction_check_chunk_frames": int(
                    ABSTRACTION_CHECK_CHUNK_FRAMES
                ),
                "min_abstraction_accept_frames": int(
                    MIN_ABSTRACTION_ACCEPT_FRAMES
                ),
                "final_single_check_below_frames": np.nan,
                "abstraction_offset_start": int(
                    offset_start
                ),
                "abstraction_offset_end": int(
                    offset_end_stored
                ),
                "abstraction_decision_offset": int(
                    offset_end_requested
                ),
                "abstraction_offset_end_requested": int(
                    offset_end_requested
                ),
                "abstraction_requested_endpoint_frame": int(
                    requested_endpoint_frame
                ),
                "abstraction_requested_endpoint_x": float(
                    requested_endpoint_xy[
                        0
                    ]
                ),
                "abstraction_requested_endpoint_y": float(
                    requested_endpoint_xy[
                        1
                    ]
                ),
                "abstraction_endpoint_error_pixels": float(
                    endpoint_error
                ),
                "abstraction_terminal_step_within_span": (
                    int(
                        terminal_step_within_span
                    )
                    if terminal_step_within_span
                    is not None
                    else np.nan
                ),
                "abstraction_terminal_fraction": (
                    float(
                        terminal_fraction
                    )
                    if np.isfinite(
                        terminal_fraction
                    )
                    else np.nan
                ),
                "first_bad_offset": np.nan,
                "simulation_stop_reason": "",
                "terminal_event": "",
            }

            hybrid_rows.append(
                {
                    **common,
                    "abstraction_line_role": "start",
                    "frame": int(
                        line_start_frame
                    ),
                    "x": float(
                        line_start_position[
                            0
                        ]
                    ),
                    "y": float(
                        line_start_position[
                            1
                        ]
                    ),
                    "predicted_error_pixels": 0.0,
                }
            )

            hybrid_rows.append(
                {
                    **common,
                    "abstraction_line_role": "endpoint",
                    "frame": int(
                        endpoint_frame
                    ),
                    "x": float(
                        endpoint_xy[
                            0
                        ]
                    ),
                    "y": float(
                        endpoint_xy[
                            1
                        ]
                    ),
                    # This is deliberately the error at the full requested
                    # A-frame endpoint, even when the stored line endpoint is
                    # an earlier terminal-contact point.
                    "predicted_error_pixels": float(
                        endpoint_error
                    ),
                    "terminal_event": (
                        terminal_event
                    ),
                }
            )

            current_frame = int(
                endpoint_frame
            )

            current_position = (
                endpoint_xy
            )

            # The decision consumed the full requested abstraction span.
            # This cursor matters only when there was no terminal event.
            offset_cursor = int(
                offset_end_requested
            )

            segment_id += 1

            if terminal_event:
                rollout_terminal_event = (
                    terminal_event
                )

                rollout_stop_reason = (
                    f"terminal_{terminal_event}"
                )

                break

        if rollout_terminal_event:
            break

        if current_frame >= int(
            MAX_TIME_ONLY_ROLLOUT_FRAMES
        ):
            rollout_stop_reason = (
                "safety_max_time_only_rollout_frames"
            )
            break

        if segment_id >= int(
            max_segments
        ):
            rollout_stop_reason = (
                "safety_max_segments"
            )
            break

        if not candidate_failed:
            rollout_stop_reason = (
                "rnn_phase_ended_without_failure"
            )
            break

        sim_start_frame = int(
            current_frame
        )

        sim_start_position = np.asarray(
            current_position,
            dtype=float,
        )

        if (
            last_straight_line_velocity
            is not None
        ):
            sim_start_velocity = np.asarray(
                last_straight_line_velocity,
                dtype=float,
            )
        else:
            sim_start_velocity = (
                estimate_velocity_for_sim(
                    history_positions,
                    [],
                )
            )

        (
            sim_df,
            sim_cache_frames,
            sim_full_images,
            stop_reason,
            adaptive_meta,
        ) = run_adaptive_simulation_segment(
            scene_dir=scene_dir,
            threshold_pixels=threshold_pixels,
            sim_start_frame=sim_start_frame,
            sim_start_position=sim_start_position,
            sim_start_velocity=sim_start_velocity,
            terminal_geometry=terminal_geometry,
        )

        sim_terminal_event = ""

        if (
            stop_reason
            == "goal_distance_threshold"
        ):
            sim_terminal_event = "goal"

        elif (
            stop_reason
            == "ground_distance_threshold"
        ):
            sim_terminal_event = "ground"

        for row_i, (
            _,
            r,
        ) in enumerate(
            sim_df.iterrows()
        ):
            is_last = (
                row_i
                == len(
                    sim_df
                )
                - 1
            )

            hybrid_rows.append(
                {
                    "scene": str(
                        scene_dir
                    ),
                    "threshold_pixels": float(
                        threshold_pixels
                    ),
                    "segment_id": int(
                        segment_id
                    ),
                    "source": "simulation",
                    "abstraction_line_role": "",
                    "frame": int(
                        r[
                            "frame"
                        ]
                    ),
                    "x": float(
                        r[
                            "x"
                        ]
                    ),
                    "y": float(
                        r[
                            "y"
                        ]
                    ),
                    "vx": float(
                        r[
                            "vx"
                        ]
                    ),
                    "vy": float(
                        r[
                            "vy"
                        ]
                    ),
                    "predicted_error_pixels": np.nan,
                    "first_bad_offset": (
                        first_bad_offset
                        if first_bad_offset
                        is not None
                        else np.nan
                    ),
                    "simulation_stop_reason": (
                        stop_reason
                        if is_last
                        else ""
                    ),
                    "terminal_event": (
                        sim_terminal_event
                        if is_last
                        else ""
                    ),
                    "adaptive_sim_requested_frames": (
                        adaptive_meta.get(
                            "adaptive_sim_requested_frames",
                            np.nan,
                        )
                    ),
                    "adaptive_sim_accepted_frames": (
                        adaptive_meta.get(
                            "adaptive_sim_accepted_frames",
                            np.nan,
                        )
                    ),
                    "adaptive_sim_num_attempts": (
                        adaptive_meta.get(
                            "adaptive_sim_num_attempts",
                            np.nan,
                        )
                    ),
                    "adaptive_sim_next_check_error_pixels": (
                        adaptive_meta.get(
                            "adaptive_sim_next_check_error_pixels",
                            np.nan,
                        )
                    ),
                    "adaptive_sim_next_check_terminal_event": (
                        adaptive_meta.get(
                            "adaptive_sim_next_check_terminal_event",
                            "",
                        )
                    ),
                    "adaptive_sim_accept_reason": (
                        adaptive_meta.get(
                            "adaptive_sim_accept_reason",
                            "",
                        )
                    ),
                }
            )

        if len(
            sim_df
        ) == 0:
            rollout_stop_reason = (
                adaptive_meta.get(
                    "adaptive_sim_accept_reason",
                    "simulation_returned_no_frames",
                )
            )
            break

        current_frame = int(
            sim_df.iloc[
                -1
            ][
                "frame"
            ]
        )

        current_position = (
            sim_df.iloc[
                -1
            ][
                [
                    "x",
                    "y",
                ]
            ]
            .to_numpy(
                dtype=float
            )
        )

        if sim_terminal_event:
            rollout_terminal_event = (
                sim_terminal_event
            )

            rollout_stop_reason = (
                f"terminal_{sim_terminal_event}"
            )

            break

        if len(
            sim_full_images
        ) < N_HISTORY:
            rollout_stop_reason = (
                "fewer_than_history_simulation_feedback_frames"
            )
            break

        history_frames = (
            pil_images_to_rnn_history_array(
                sim_full_images[
                    -N_HISTORY:
                ]
            )
        )

        validate_rnn_history_array(
            history_frames,
            context=(
                "simulation feedback history"
            ),
        )

        history_positions = (
            sim_df[
                [
                    "x",
                    "y",
                ]
            ]
            .to_numpy(
                dtype=float
            )
            .tolist()[
                -N_HISTORY:
            ]
        )

        segment_id += 1

    hybrid_df = pd.DataFrame(
        hybrid_rows
    )

    if len(
        hybrid_df
    ) > 0:
        hybrid_df[
            "rollout_terminal_event"
        ] = rollout_terminal_event

        hybrid_df[
            "rollout_stop_reason"
        ] = rollout_stop_reason

        true_table = pos_df.copy()

        if (
            "frame_index"
            in true_table.columns
        ):
            true_table = (
                true_table.set_index(
                    "frame_index"
                )
            )
        else:
            true_table.index = np.arange(
                len(
                    true_table
                )
            )

        true_x = []
        true_y = []

        for frame in hybrid_df[
            "frame"
        ].astype(
            int
        ):
            if frame in true_table.index:
                row = true_table.loc[
                    frame
                ]

                true_x.append(
                    float(
                        row[
                            "ball_x"
                        ]
                    )
                )

                true_y.append(
                    float(
                        row[
                            "ball_y"
                        ]
                    )
                )
            else:
                true_x.append(
                    np.nan
                )

                true_y.append(
                    np.nan
                )

        hybrid_df[
            "true_x"
        ] = true_x

        hybrid_df[
            "true_y"
        ] = true_y

        hybrid_df[
            "euclidean_error_pixels"
        ] = np.sqrt(
            (
                hybrid_df[
                    "x"
                ]
                - hybrid_df[
                    "true_x"
                ]
            )
            ** 2
            + (
                hybrid_df[
                    "y"
                ]
                - hybrid_df[
                    "true_y"
                ]
            )
            ** 2
        )

    return hybrid_df




In [ ]:
# ============================================================
# 6. Metric helpers and serial preflight
# ============================================================
def count_abstraction_steps(
    hybrid_df,
):
    """
    Count each accepted straight-line abstraction segment once.
    """
    if (
        hybrid_df is None
        or hybrid_df.empty
    ):
        return 0

    abstraction_df = hybrid_df.loc[
        hybrid_df[
            "source"
        ]
        .astype(str)
        .eq(
            "abstraction"
        )
    ]

    if abstraction_df.empty:
        return 0

    if (
        "abstraction_line_role"
        in abstraction_df.columns
    ):
        endpoint_count = int(
            abstraction_df[
                "abstraction_line_role"
            ]
            .astype(str)
            .str.strip()
            .str.lower()
            .eq(
                "endpoint"
            )
            .sum()
        )

        if endpoint_count > 0:
            return endpoint_count

    if (
        "segment_id"
        in abstraction_df.columns
    ):
        return int(
            abstraction_df[
                "segment_id"
            ]
            .dropna()
            .nunique()
        )

    raise RuntimeError(
        "Could not identify abstraction segments."
    )


def count_simulation_steps(
    hybrid_df,
):
    """
    Count each stored Pymunk simulation frame as one step.
    """
    if (
        hybrid_df is None
        or hybrid_df.empty
    ):
        return 0

    return int(
        hybrid_df[
            "source"
        ]
        .astype(str)
        .eq(
            "simulation"
        )
        .sum()
    )


def hybrid_pred_hit_from_df(
    hybrid_df,
):
    """
    Return 1 for predicted goal and 0 for predicted ground.
    """
    if (
        hybrid_df is None
        or hybrid_df.empty
    ):
        raise RuntimeError(
            "The hybrid rollout returned no rows."
        )

    for event_column in [
        "rollout_terminal_event",
        "terminal_event",
    ]:
        if (
            event_column
            not in hybrid_df.columns
        ):
            continue

        events = (
            hybrid_df[
                event_column
            ]
            .dropna()
            .astype(str)
            .str.strip()
            .str.lower()
        )

        events = events[
            events.isin(
                [
                    "goal",
                    "ground",
                ]
            )
        ]

        if len(events):
            return int(
                events.iloc[-1]
                == "goal"
            )

    if (
        "simulation_stop_reason"
        in hybrid_df.columns
    ):
        stop_reasons = set(
            hybrid_df[
                "simulation_stop_reason"
            ]
            .dropna()
            .astype(str)
        )

        if (
            "goal_distance_threshold"
            in stop_reasons
        ):
            return 1

        if (
            "ground_distance_threshold"
            in stop_reasons
        ):
            return 0

    raise RuntimeError(
        "The rollout did not reach a recognized "
        "goal/ground terminal event."
    )


def parse_bool_value(
    value,
):
    """
    Robustly parse CSV boolean-like values.
    """
    if isinstance(
        value,
        (
            bool,
            np.bool_,
        ),
    ):
        return bool(
            value
        )

    if pd.isna(
        value
    ):
        raise ValueError(
            "Cannot parse a missing boolean value."
        )

    if isinstance(
        value,
        (
            int,
            np.integer,
            float,
            np.floating,
        ),
    ):
        return bool(
            int(
                value
            )
        )

    text = (
        str(
            value
        )
        .strip()
        .lower()
    )

    if text in {
        "true",
        "1",
        "yes",
        "y",
    }:
        return True

    if text in {
        "false",
        "0",
        "no",
        "n",
    }:
        return False

    raise ValueError(
        f"Cannot interpret as boolean: {value!r}"
    )


def load_true_hit(
    scene_dir,
):
    """
    Return 1 for a recorded goal hit and 0 for a ground-only ending.
    """
    scene_dir = Path(
        scene_dir
    )

    simulation_path = (
        scene_dir
        / "simulation_dataset.csv"
    )

    if not simulation_path.exists():
        raise FileNotFoundError(
            simulation_path
        )

    scene_df = pd.read_csv(
        simulation_path,
        nrows=1,
    )

    if scene_df.empty:
        raise RuntimeError(
            f"Empty simulation data: {simulation_path}"
        )

    row = scene_df.iloc[0]

    if (
        "goal_hit_ever"
        in scene_df.columns
        and pd.notna(
            row[
                "goal_hit_ever"
            ]
        )
    ):
        return int(
            parse_bool_value(
                row[
                    "goal_hit_ever"
                ]
            )
        )

    for event_column in [
        "terminal_event_type",
        "stop_event_type",
    ]:
        if (
            event_column
            not in scene_df.columns
            or pd.isna(
                row[
                    event_column
                ]
            )
        ):
            continue

        event = (
            str(
                row[
                    event_column
                ]
            )
            .strip()
            .lower()
        )

        if event in {
            "goal",
            "goal_and_ground",
        }:
            return 1

        if event == "ground":
            return 0

    if (
        "goal_hit_frame"
        in scene_df.columns
        and pd.notna(
            row[
                "goal_hit_frame"
            ]
        )
    ):
        return 1

    raise RuntimeError(
        f"Could not determine true_hit for {scene_dir}."
    )


def normalize_scene_name(
    value,
):
    """
    Normalize a scene name for a strict name-based join.
    """
    name = Path(
        str(
            value
        ).strip()
    ).name

    if name.lower().endswith(
        ".json"
    ):
        name = name[:-5]

    return name.casefold()


def find_human_scene_column(
    dataframe,
):
    for column in [
        "scene_name",
        "scene",
        "scene_id",
        "json_name",
        "trial_name",
        "trial",
    ]:
        if column in dataframe.columns:
            return column

    raise KeyError(
        "Could not find a scene-name column in "
        f"{HUMAN_SUMMARY_PATH}. Available columns: "
        f"{list(dataframe.columns)}"
    )


human_summary = pd.read_csv(
    HUMAN_SUMMARY_PATH
)

required_human_columns = {
    "mean_accuracy",
    "mean_judgment_time",
}

missing_human_columns = (
    required_human_columns
    - set(
        human_summary.columns
    )
)

if missing_human_columns:
    raise KeyError(
        "Human summary is missing columns: "
        f"{sorted(missing_human_columns)}"
    )


human_scene_column = (
    find_human_scene_column(
        human_summary
    )
)

human_lookup_df = human_summary[
    [
        human_scene_column,
        "mean_accuracy",
        "mean_judgment_time",
    ]
].copy()

human_lookup_df[
    "_scene_key"
] = human_lookup_df[
    human_scene_column
].map(
    normalize_scene_name
)

human_lookup_df[
    "mean_accuracy"
] = pd.to_numeric(
    human_lookup_df[
        "mean_accuracy"
    ],
    errors="coerce",
)

human_lookup_df[
    "mean_judgment_time"
] = pd.to_numeric(
    human_lookup_df[
        "mean_judgment_time"
    ],
    errors="coerce",
)


duplicate_human_keys = (
    human_lookup_df[
        "_scene_key"
    ]
    .duplicated(
        keep=False
    )
)

if duplicate_human_keys.any():
    duplicate_table = human_lookup_df.loc[
        duplicate_human_keys
    ]

    conflicting_keys = []

    for scene_key, group in duplicate_table.groupby(
        "_scene_key"
    ):
        if (
            group[
                "mean_accuracy"
            ]
            .dropna()
            .nunique()
            > 1
            or group[
                "mean_judgment_time"
            ]
            .dropna()
            .nunique()
            > 1
        ):
            conflicting_keys.append(
                scene_key
            )

    if conflicting_keys:
        raise RuntimeError(
            "Conflicting duplicate human rows for "
            f"scene keys: {conflicting_keys}"
        )

human_lookup_df = (
    human_lookup_df
    .drop_duplicates(
        "_scene_key"
    )
    .set_index(
        "_scene_key"
    )
)


scene_summary_df = pd.read_csv(SCENE_SUMMARY_PATH)
scene_summary_name_column = find_human_scene_column(scene_summary_df)
if "simulation_time" not in scene_summary_df.columns:
    raise KeyError(
        f"{SCENE_SUMMARY_PATH} must contain a simulation_time column."
    )
simulation_time_lookup = scene_summary_df[
    [scene_summary_name_column, "simulation_time"]
].copy()
simulation_time_lookup["_scene_key"] = simulation_time_lookup[
    scene_summary_name_column
].map(normalize_scene_name)
simulation_time_lookup["simulation_time"] = pd.to_numeric(
    simulation_time_lookup["simulation_time"],
    errors="coerce",
)
duplicate_sim_keys = simulation_time_lookup["_scene_key"].duplicated(
    keep=False
)
if duplicate_sim_keys.any():
    conflicting_sim_keys = []
    for scene_key, group in simulation_time_lookup.loc[
        duplicate_sim_keys
    ].groupby("_scene_key"):
        if group["simulation_time"].dropna().nunique() > 1:
            conflicting_sim_keys.append(scene_key)
    if conflicting_sim_keys:
        raise RuntimeError(
            "Conflicting simulation_time values for scene keys: "
            f"{conflicting_sim_keys}"
        )
simulation_time_lookup = (
    simulation_time_lookup
    .drop_duplicates("_scene_key")
    .set_index("_scene_key")
)


test_scene_keys = [
    normalize_scene_name(
        scene_dir.name
    )
    for scene_dir in test_scene_dirs
]

if len(
    set(
        test_scene_keys
    )
) != len(
    test_scene_keys
):
    raise RuntimeError(
        "Testing scene names are not unique after "
        "normalization, so a name-only human join "
        "would be ambiguous."
    )


# Build missing tiny temporary JSONs and verify all large
# frame/position caches serially before worker processes start.
SCENE_JSON_PATH_LOOKUP = {}
GRID_SCENE_RECORDS = []
unmatched_human_scenes = []


for scene_index, scene_dir in enumerate(
    test_scene_dirs,
    start=1,
):
    scene_dir = Path(
        scene_dir
    )

    frame_path = frame_cache_path(
        scene_dir
    )

    position_path = ball_cache_path(
        scene_dir
    )

    if not frame_path.exists():
        raise FileNotFoundError(
            "Missing frame cache before grid search: "
            f"{frame_path}"
        )

    if not position_path.exists():
        raise FileNotFoundError(
            "Missing ball-position cache before grid search: "
            f"{position_path}"
        )

    frame_cache = np.load(
        frame_path,
        mmap_mode="r",
    )

    if len(
        frame_cache
    ) < N_HISTORY:
        raise RuntimeError(
            f"{scene_dir} has only {len(frame_cache)} "
            "cached frames."
        )

    del frame_cache

    position_preview = pd.read_csv(
        position_path,
        nrows=N_HISTORY,
    )

    if len(
        position_preview
    ) < N_HISTORY:
        raise RuntimeError(
            f"{scene_dir} has only "
            f"{len(position_preview)} cached positions."
        )

    json_path = get_json_for_scene(
        scene_dir
    )

    SCENE_JSON_PATH_LOOKUP[
        str(
            scene_dir.resolve()
        )
    ] = str(
        Path(
            json_path
        ).resolve()
    )

    scene_key = normalize_scene_name(
        scene_dir.name
    )

    true_hit = load_true_hit(
        scene_dir
    )
    
    if scene_key not in simulation_time_lookup.index:
        raise RuntimeError(
            f"No simulation_time match in {SCENE_SUMMARY_PATH} "
            f"for scene {scene_dir.name!r}."
        )

    simulation_time = float(
        simulation_time_lookup.loc[
            scene_key,
            "simulation_time",
        ]
    )
    if not np.isfinite(simulation_time):
        raise RuntimeError(
            f"Non-finite simulation_time for scene {scene_dir.name!r}."
        )

    if scene_key in human_lookup_df.index:
        human_row = human_lookup_df.loc[
            scene_key
        ]

        mean_accuracy = float(
            human_row[
                "mean_accuracy"
            ]
        )

        mean_judgment_time = float(
            human_row[
                "mean_judgment_time"
            ]
        )

    else:
        mean_accuracy = np.nan
        mean_judgment_time = np.nan

        unmatched_human_scenes.append(
            scene_dir.name
        )

    GRID_SCENE_RECORDS.append(
        {
            "scene_dir": str(
                scene_dir.resolve()
            ),
            "scene_name": (
                scene_dir.name
            ),
            "true_hit": int(
                true_hit
            ),
            "simulation_time": simulation_time,
            "mean_accuracy": (
                mean_accuracy
            ),
            "mean_judgment_time": (
                mean_judgment_time
            ),
        }
    )

    if (
        scene_index % 20 == 0
        or scene_index
        == len(
            test_scene_dirs
        )
    ):
        print(
            "Preflight:",
            f"{scene_index}/"
            f"{len(test_scene_dirs)} scenes",
        )


print(
    "Human scene-name column:",
    human_scene_column,
)

print(
    "Testing scenes prepared:",
    len(
        GRID_SCENE_RECORDS
    ),
)

print(
    "Scenes with finite human accuracy and RT:",
    sum(
        np.isfinite(
            record[
                "mean_accuracy"
            ]
        )
        and np.isfinite(
            record[
                "mean_judgment_time"
            ]
        )
        for record in GRID_SCENE_RECORDS
    ),
)

if unmatched_human_scenes:
    print(
        "Scenes without a human-summary name match "
        "(excluded only from human-based metrics):"
    )

    for scene_name in unmatched_human_scenes:
        print(
            " ",
            scene_name,
        )


In [ ]:
# ============================================================
# 7. Run the 275-combination grid search serially
# ============================================================
def _safe_raw_pearson_r(
    total_steps,
    judgment_time,
):
    """
    Calculate Pearson r between original total hybrid steps and
    original mean human judgment time.

    Only observations with finite values are included.
    """
    total_steps = np.asarray(
        total_steps,
        dtype=float,
    )

    judgment_time = np.asarray(
        judgment_time,
        dtype=float,
    )

    valid = (
        np.isfinite(
            total_steps
        )
        & np.isfinite(
            judgment_time
        )
    )

    total_steps = total_steps[
        valid
    ]

    judgment_time = judgment_time[
        valid
    ]

    if len(
        total_steps
    ) < 2:
        return np.nan

    if (
        np.allclose(
            total_steps,
            total_steps[
                0
            ],
        )
        or np.allclose(
            judgment_time,
            judgment_time[
                0
            ],
        )
    ):
        return np.nan

    return float(
        pearsonr(
            total_steps,
            judgment_time,
        ).statistic
    )


def _safe_log_log_pearson_r(
    total_steps,
    judgment_time,
):
    """
    Calculate Pearson r between log10(total hybrid steps) and
    log10(mean human judgment time).

    Only observations with finite, strictly positive original
    values are included.
    """
    total_steps = np.asarray(
        total_steps,
        dtype=float,
    )

    judgment_time = np.asarray(
        judgment_time,
        dtype=float,
    )

    valid = (
        np.isfinite(
            total_steps
        )
        & np.isfinite(
            judgment_time
        )
        & (
            total_steps
            > 0
        )
        & (
            judgment_time
            > 0
        )
    )

    total_steps = total_steps[
        valid
    ]

    judgment_time = judgment_time[
        valid
    ]

    if len(
        total_steps
    ) < 2:
        return np.nan

    log_total_steps = np.log10(
        total_steps
    )

    log_judgment_time = np.log10(
        judgment_time
    )

    if (
        np.allclose(
            log_total_steps,
            log_total_steps[
                0
            ],
        )
        or np.allclose(
            log_judgment_time,
            log_judgment_time[
                0
            ],
        )
    ):
        return np.nan

    return float(
        pearsonr(
            log_total_steps,
            log_judgment_time,
        ).statistic
    )


def _evaluate_parameter_combination(
    parameter_tuple,
):
    """
    Evaluate one (A, S, e) combination across every testing scene.

    Returns only compact aggregate metrics; no rollout DataFrames,
    plots, or CSVs are saved.
    """
    global ABSTRACTION_CHECK_CHUNK_FRAMES
    global SIM_CHUNK_FRAMES
    global ADAPTIVE_SIM_INCREMENT_FRAMES

    A, S, error_threshold = (
        parameter_tuple
    )

    ABSTRACTION_CHECK_CHUNK_FRAMES = int(
        A
    )

    SIM_CHUNK_FRAMES = int(
        S
    )

    ADAPTIVE_SIM_INCREMENT_FRAMES = int(
        S
    )

    scene_rows = []

    for scene_number, record in enumerate(
        GRID_SCENE_RECORDS,
        start=1,
    ):
        scene_dir = Path(
            record[
                "scene_dir"
            ]
        )

        try:
            hybrid_df = run_hybrid_for_scene(
                scene_dir,
                threshold_pixels=float(
                    error_threshold
                ),
                max_segments=(
                    MAX_SEGMENTS_PER_SCENE
                ),
            )

            if hybrid_df.empty:
                raise RuntimeError(
                    "Hybrid rollout returned no rows."
                )

            abstraction_steps = (
                count_abstraction_steps(
                    hybrid_df
                )
            )

            simulation_steps = (
                count_simulation_steps(
                    hybrid_df
                )
            )

            total_steps = int(
                abstraction_steps
                + simulation_steps
            )

            hybrid_pred_hit = (
                hybrid_pred_hit_from_df(
                    hybrid_df
                )
            )

            hybrid_correct = int(
                hybrid_pred_hit
                == int(
                    record[
                        "true_hit"
                    ]
                )
            )

            scene_rows.append(
                {
                    "scene_name": (
                        record[
                            "scene_name"
                        ]
                    ),
                    "total_step_count": (
                        total_steps
                    ),
                    "hybrid_correct": (
                        hybrid_correct
                    ),
                    "mean_accuracy": (
                        record[
                            "mean_accuracy"
                        ]
                    ),
                    "mean_judgment_time": (
                        record[
                            "mean_judgment_time"
                        ]
                    ),
                }
            )

            del hybrid_df

        except Exception as error:
            return {
                "A": int(
                    A
                ),
                "S": int(
                    S
                ),
                "e": float(
                    error_threshold
                ),
                "status": "failed",
                "n_scenes": len(
                    GRID_SCENE_RECORDS
                ),
                "n_completed_scenes": (
                    scene_number
                    - 1
                ),
                "n_time_correlation_scenes": 0,
                "n_median_accuracy_scenes": 0,
                "n_hybrid_correct": 0,
                "n_hybrid_incorrect": 0,
                "overall_accuracy": np.nan,
                "raw_step_time_correlation_r": np.nan,
                "log_log_step_time_correlation_r": np.nan,
                "median_human_accuracy_correct": np.nan,
                "median_human_accuracy_incorrect": np.nan,
                "median_human_accuracy_difference": np.nan,
                "error": (
                    f"{record['scene_name']}: "
                    f"{type(error).__name__}: "
                    f"{error}"
                )[
                    :500
                ],
            }

    scene_metrics = pd.DataFrame(
        scene_rows
    )

    n_scenes = int(
        len(
            scene_metrics
        )
    )

    n_hybrid_correct = int(
        scene_metrics[
            "hybrid_correct"
        ].sum()
    )

    n_hybrid_incorrect = int(
        n_scenes
        - n_hybrid_correct
    )

    overall_accuracy = float(
        scene_metrics[
            "hybrid_correct"
        ].mean()
    )

    scene_name_text = (
        scene_metrics[
            "scene_name"
        ]
        .astype(str)
        .str.strip()
    )

    starts_with_scene_mask = (
        scene_name_text
        .str.startswith(
            "scene",
            na=False,
        )
    )

    # The two time correlations use only the 48 testing scenes
    # whose names do NOT begin with "scene".
    time_scene_group_mask = (
        ~starts_with_scene_mask
    )

    # The median human-accuracy comparison uses only the 58
    # testing scenes whose names begin with "scene".
    accuracy_scene_group_mask = (
        starts_with_scene_mask
    )

    n_time_scene_group = int(
        time_scene_group_mask.sum()
    )

    n_accuracy_scene_group = int(
        accuracy_scene_group_mask.sum()
    )

    if n_time_scene_group != 48:
        raise RuntimeError(
            "Expected exactly 48 testing scenes whose names "
            'do not start with "scene", but found '
            f"{n_time_scene_group}."
        )

    if n_accuracy_scene_group != 58:
        raise RuntimeError(
            "Expected exactly 58 testing scenes whose names "
            'start with "scene", but found '
            f"{n_accuracy_scene_group}."
        )

    time_metric_mask = (
        time_scene_group_mask
        & np.isfinite(
            scene_metrics[
                "total_step_count"
            ].to_numpy(
                dtype=float
            )
        )
        & np.isfinite(
            scene_metrics[
                "mean_judgment_time"
            ].to_numpy(
                dtype=float
            )
        )
    )

    accuracy_metric_mask = (
        accuracy_scene_group_mask
        & np.isfinite(
            scene_metrics[
                "mean_accuracy"
            ].to_numpy(
                dtype=float
            )
        )
    )

    time_metrics = (
        scene_metrics.loc[
            time_metric_mask
        ]
        .copy()
    )

    accuracy_metrics = (
        scene_metrics.loc[
            accuracy_metric_mask
        ]
        .copy()
    )

    n_time_correlation_scenes = int(
        len(
            time_metrics
        )
    )

    n_median_accuracy_scenes = int(
        len(
            accuracy_metrics
        )
    )

    raw_step_time_correlation_r = (
        _safe_raw_pearson_r(
            time_metrics[
                "total_step_count"
            ],
            time_metrics[
                "mean_judgment_time"
            ],
        )
    )

    log_log_step_time_correlation_r = (
        _safe_log_log_pearson_r(
            time_metrics[
                "total_step_count"
            ],
            time_metrics[
                "mean_judgment_time"
            ],
        )
    )

    correct_human_accuracy = (
        accuracy_metrics.loc[
            accuracy_metrics[
                "hybrid_correct"
            ].eq(
                1
            ),
            "mean_accuracy",
        ]
        .dropna()
    )

    incorrect_human_accuracy = (
        accuracy_metrics.loc[
            accuracy_metrics[
                "hybrid_correct"
            ].eq(
                0
            ),
            "mean_accuracy",
        ]
        .dropna()
    )

    median_human_accuracy_correct = (
        float(
            correct_human_accuracy.median()
        )
        if len(
            correct_human_accuracy
        )
        else np.nan
    )

    median_human_accuracy_incorrect = (
        float(
            incorrect_human_accuracy.median()
        )
        if len(
            incorrect_human_accuracy
        )
        else np.nan
    )

    if (
        np.isfinite(
            median_human_accuracy_correct
        )
        and np.isfinite(
            median_human_accuracy_incorrect
        )
    ):
        median_human_accuracy_difference = float(
            median_human_accuracy_correct
            - median_human_accuracy_incorrect
        )
    else:
        median_human_accuracy_difference = np.nan

    return {
        "A": int(
            A
        ),
        "S": int(
            S
        ),
        "e": float(
            error_threshold
        ),
        "status": "ok",
        "n_scenes": n_scenes,
        "n_completed_scenes": n_scenes,
        "n_time_correlation_scenes": (
            n_time_correlation_scenes
        ),
        "n_median_accuracy_scenes": (
            n_median_accuracy_scenes
        ),
        "n_hybrid_correct": (
            n_hybrid_correct
        ),
        "n_hybrid_incorrect": (
            n_hybrid_incorrect
        ),
        "overall_accuracy": (
            overall_accuracy
        ),
        "raw_step_time_correlation_r": (
            raw_step_time_correlation_r
        ),
        "log_log_step_time_correlation_r": (
            log_log_step_time_correlation_r
        ),
        "median_human_accuracy_correct": (
            median_human_accuracy_correct
        ),
        "median_human_accuracy_incorrect": (
            median_human_accuracy_incorrect
        ),
        "median_human_accuracy_difference": (
            median_human_accuracy_difference
        ),
        "error": "",
    }


def _format_metric(
    value,
    digits=4,
):
    if value is None:
        return "NA"

    try:
        value = float(
            value
        )
    except Exception:
        return "NA"

    if not np.isfinite(
        value
    ):
        return "NA"

    return f"{value:.{digits}f}"


def _print_parameter_result(
    result,
    completed_index,
    total_count,
):
    prefix = (
        f"[{completed_index:03d}/"
        f"{total_count:03d}] "
        f"A={result['A']:3d}, "
        f"S={result['S']:2d}, "
        f"e={result['e']:4.1f}"
    )

    if (
        result[
            "status"
        ]
        == "ok"
    ):
        print(
            prefix
            + " | accuracy="
            + _format_metric(
                result[
                    "overall_accuracy"
                ]
            )
            + " | raw step-time r="
            + _format_metric(
                result[
                    "raw_step_time_correlation_r"
                ]
            )
            + " | log-log step-time r="
            + _format_metric(
                result[
                    "log_log_step_time_correlation_r"
                ]
            )
            + " | median human-accuracy diff="
            + _format_metric(
                result[
                    "median_human_accuracy_difference"
                ]
            )
            + " | time n="
            + str(
                result[
                    "n_time_correlation_scenes"
                ]
            )
            + " | accuracy n="
            + str(
                result[
                    "n_median_accuracy_scenes"
                ]
            )
        )

    else:
        print(
            prefix
            + " | FAILED after "
            + str(
                result[
                    "n_completed_scenes"
                ]
            )
            + " scenes | "
            + str(
                result[
                    "error"
                ]
            )
        )


def run_parameter_grid():
    """
    Run every parameter combination strictly serially.

    One compact result line is printed immediately after each
    combination finishes.
    """
    total_count = len(
        PARAMETER_GRID
    )

    results = []

    print(
        "\nRunning all parameter combinations serially."
    )

    for completed_index, parameters in enumerate(
        PARAMETER_GRID,
        start=1,
    ):
        result = (
            _evaluate_parameter_combination(
                parameters
            )
        )

        results.append(
            result
        )

        _print_parameter_result(
            result,
            completed_index,
            total_count,
        )

    return pd.DataFrame(
        results
    )


grid_start_time = time.time()

grid_results = (
    run_parameter_grid()
)

grid_results = (
    grid_results
    .sort_values(
        [
            "A",
            "S",
            "e",
        ]
    )
    .reset_index(
        drop=True
    )
)


expected_parameter_rows = len(
    PARAMETER_GRID
)

if len(
    grid_results
) != expected_parameter_rows:
    raise RuntimeError(
        "Expected one grid-result row per parameter "
        f"combination ({expected_parameter_rows}), but "
        f"found {len(grid_results)}."
    )


duplicate_parameter_rows = (
    grid_results[
        [
            "A",
            "S",
            "e",
        ]
    ]
    .duplicated()
)

if duplicate_parameter_rows.any():
    raise RuntimeError(
        "Duplicate parameter combinations appear in "
        "the final grid table."
    )


GRID_RESULTS_PATH = (
    GRID_OUTPUT_DIR
    / "grid_search_results.csv"
)
grid_results.to_csv(
    GRID_RESULTS_PATH,
    index=False,
)

elapsed_minutes = (
    time.time()
    - grid_start_time
) / 60.0


print(
    "\n"
    + "="
    * 125
)

print(
    "FINAL GRID RESULTS"
)

print(
    "Raw step-time r is Pearson correlation between "
    "total_step_count and mean_judgment_time using only "
    'the 48 scenes whose names do not start with "scene".'
)

print(
    "Log-log step-time r is Pearson correlation between "
    "log10(total_step_count) and "
    "log10(mean_judgment_time), using the same 48 "
    'non-"scene" scenes.'
)

print(
    "Median human-accuracy difference is correct-scene "
    "median minus incorrect-scene median using only the "
    '58 scenes whose names start with "scene".'
)

print(
    "Grid results saved to:",
    GRID_RESULTS_PATH,
)

print(
    f"Elapsed minutes: {elapsed_minutes:.2f}"
)

print(
    "="
    * 125
)


print_columns = [
    "A",
    "S",
    "e",
    "overall_accuracy",
    "raw_step_time_correlation_r",
    "log_log_step_time_correlation_r",
    "median_human_accuracy_difference",
    "n_time_correlation_scenes",
    "n_median_accuracy_scenes",
    "n_hybrid_correct",
    "n_hybrid_incorrect",
    "status",
]


print(
    grid_results[
        print_columns
    ].to_string(
        index=False,
        float_format=lambda value: (
            f"{value:.4f}"
        ),
    )
)


## Selected A150/S20/e22 meta-control rollout

In [ ]:
# ============================================================
# 7. Overlay plotting and summary helpers
# ============================================================

def _standardize_hybrid_plot_df(hybrid_df):
    """
    Return a time-ordered hybrid trajectory DataFrame with standardized columns:
      frame, x, y, source
    """
    df = hybrid_df.copy()

    if len(df) == 0:
        return df

    if "frame" not in df.columns:
        if "target_frame" in df.columns:
            df["frame"] = df["target_frame"]
        elif "time_step" in df.columns:
            df["frame"] = df["time_step"]
        elif "t" in df.columns:
            df["frame"] = df["t"]
        else:
            raise RuntimeError(
                "hybrid_df must contain frame, target_frame, time_step, or t."
            )

    if "x" not in df.columns:
        if "hybrid_x" in df.columns:
            df["x"] = df["hybrid_x"]
        elif "pred_x" in df.columns:
            df["x"] = df["pred_x"]
        else:
            raise RuntimeError("hybrid_df must contain x, hybrid_x, or pred_x.")

    if "y" not in df.columns:
        if "hybrid_y" in df.columns:
            df["y"] = df["hybrid_y"]
        elif "pred_y" in df.columns:
            df["y"] = df["pred_y"]
        else:
            raise RuntimeError("hybrid_df must contain y, hybrid_y, or pred_y.")

    if "source" not in df.columns:
        if "segment_type" in df.columns:
            df["source"] = df["segment_type"]
        elif "mode" in df.columns:
            df["source"] = df["mode"]
        elif "prediction_type" in df.columns:
            df["source"] = df["prediction_type"]
        else:
            raise RuntimeError(
                "hybrid_df must contain source, segment_type, mode, or prediction_type."
            )

    df["source"] = df["source"].astype(str).str.lower()
    df.loc[df["source"].str.contains("abstract"), "source"] = "abstraction"
    df.loc[df["source"].str.contains("sim"), "source"] = "simulation"
    df["frame"] = df["frame"].astype(int)

    sort_tiebreaker = "segment_id" if "segment_id" in df.columns else "source"
    df = (
        df.sort_values(["frame", sort_tiebreaker])
        .drop_duplicates(subset=["frame"], keep="last")
        .sort_values("frame")
        .reset_index(drop=True)
    )

    return df


def plot_hybrid_overlay(scene_dir, hybrid_df, out_path):
    """
    Overlay the final hybrid trajectory on the scene.

    Visualization rules:
    - true trajectory: red, underneath the hybrid trajectory;
    - RNN abstraction: coral;
    - adjacent abstraction points are connected unless simulation rows intervene;
    - Pymunk simulation: green, connected only across consecutive simulation frames;
    - no blue endpoint/goal rectangles;
    - no legend.
    """
    scene_dir = Path(scene_dir)
    pos_df = load_scene_positions(scene_dir)

    bg_path = scene_dir / "frames" / f"frame_{N_HISTORY - 1:04d}.png"
    if not bg_path.exists():
        frame_files = get_frame_files(scene_dir)
        bg_path = frame_files[min(N_HISTORY - 1, len(frame_files) - 1)]

    img = Image.open(bg_path).convert("RGB")

    fig, ax = plt.subplots(figsize=(8, 10))
    ax.imshow(img, zorder=0)

    color_map = {
        "abstraction": "coral",
        "simulation": "green",
    }

    ordered = (
        _standardize_hybrid_plot_df(hybrid_df)
        if len(hybrid_df) > 0
        else None
    )

    true_xy = pos_df[["ball_x", "ball_y"]].to_numpy(dtype=float)
    ax.plot(
        true_xy[:, 0],
        true_xy[:, 1],
        color="red",
        linewidth=2.2,
        alpha=0.85,
        zorder=2,
    )
    ax.scatter(
        true_xy[:, 0],
        true_xy[:, 1],
        s=5,
        color="red",
        alpha=0.65,
        zorder=3,
    )

    if ordered is not None and len(ordered) > 0:
        for i in range(len(ordered) - 1):
            a = ordered.iloc[i]
            b = ordered.iloc[i + 1]
            source_a = str(a["source"])
            source_b = str(b["source"])

            if source_a == "abstraction" and source_b == "abstraction":
                ax.plot(
                    [float(a["x"]), float(b["x"])],
                    [float(a["y"]), float(b["y"])],
                    color=color_map["abstraction"],
                    linewidth=2.4,
                    alpha=0.90,
                    zorder=5,
                )

            elif source_a == "simulation" and source_b == "simulation":
                if int(b["frame"]) == int(a["frame"]) + 1:
                    ax.plot(
                        [float(a["x"]), float(b["x"])],
                        [float(a["y"]), float(b["y"])],
                        color=color_map["simulation"],
                        linewidth=2.0,
                        alpha=0.88,
                        zorder=5,
                    )

        for source, group in ordered.groupby("source", sort=False):
            ax.scatter(
                group["x"],
                group["y"],
                s=22,
                color=color_map.get(source, "black"),
                alpha=0.96,
                edgecolors="black",
                linewidths=0.30,
                zorder=6,
            )

    ax.set_xlim(0, ORIGINAL_FRAME_WIDTH)
    ax.set_ylim(ORIGINAL_FRAME_HEIGHT, 0)
    threshold_text = (
        hybrid_df["threshold_pixels"].iloc[0]
        if len(hybrid_df)
        else "NA"
    )
    ax.set_title(f"{scene_dir.name} | threshold={threshold_text} px")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.grid(False)

    fig.tight_layout()
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, dpi=150)
    plt.close(fig)


def summarize_hybrid_df(hybrid_df):
    if len(hybrid_df) == 0:
        return {
            "n_points": 0,
            "mean_error_pixels": np.nan,
            "median_error_pixels": np.nan,
            "max_error_pixels": np.nan,
            "final_error_pixels": np.nan,
            "n_abstraction_points": 0,
            "n_simulation_points": 0,
            "n_abstraction_segments": 0,
            "terminal_event": "",
            "rollout_stop_reason": "",
            "reached_goal_or_ground": False,
        }

    err = hybrid_df["euclidean_error_pixels"].dropna().astype(float)

    terminal_event = ""
    if "rollout_terminal_event" in hybrid_df.columns:
        vals = hybrid_df["rollout_terminal_event"].dropna().astype(str)
        vals = vals[vals.str.len() > 0]
        if len(vals):
            terminal_event = vals.iloc[-1]
    elif "terminal_event" in hybrid_df.columns:
        vals = hybrid_df["terminal_event"].dropna().astype(str)
        vals = vals[vals.str.len() > 0]
        if len(vals):
            terminal_event = vals.iloc[-1]

    rollout_stop_reason = ""
    if "rollout_stop_reason" in hybrid_df.columns:
        vals = hybrid_df["rollout_stop_reason"].dropna().astype(str)
        vals = vals[vals.str.len() > 0]
        if len(vals):
            rollout_stop_reason = vals.iloc[-1]

    if "abstraction_line_role" in hybrid_df.columns:
        n_abstraction_segments = int(
            (
                (hybrid_df["source"] == "abstraction")
                & (
                    hybrid_df["abstraction_line_role"]
                    .astype(str)
                    .eq("endpoint")
                )
            ).sum()
        )
    else:
        n_abstraction_segments = int(
            hybrid_df.loc[
                hybrid_df["source"] == "abstraction",
                "segment_id",
            ].nunique()
        )

    return {
        "n_points": int(len(hybrid_df)),
        "mean_error_pixels": float(err.mean()) if len(err) else np.nan,
        "median_error_pixels": float(err.median()) if len(err) else np.nan,
        "max_error_pixels": float(err.max()) if len(err) else np.nan,
        "final_error_pixels": (
            float(
                hybrid_df.sort_values("frame")["euclidean_error_pixels"]
                .dropna()
                .iloc[-1]
            )
            if len(err)
            else np.nan
        ),
        "n_abstraction_points": int(
            (hybrid_df["source"] == "abstraction").sum()
        ),
        "n_simulation_points": int(
            (hybrid_df["source"] == "simulation").sum()
        ),
        "n_abstraction_segments": n_abstraction_segments,
        "terminal_event": terminal_event,
        "rollout_stop_reason": rollout_stop_reason,
        "reached_goal_or_ground": terminal_event in {"goal", "ground"},
    }


In [ ]:
# ============================================================
# Selected meta-control export
# ============================================================
PARAMETER_COMBINATIONS = [
    {
        "A": SELECTED_A,
        "S": SELECTED_S,
        "e": SELECTED_E,
        "label": SELECTED_PARAMETER_LABEL,
    }
]

HYBRID_ROOT = META_CONTROL_ROOT
SCENE_RECORDS = GRID_SCENE_RECORDS

# Match the earlier selected-run notebook behavior by recreating
# the selected-model output root before saving predictions.
if HYBRID_ROOT.exists():
    shutil.rmtree(HYBRID_ROOT)
HYBRID_ROOT.mkdir(parents=True, exist_ok=True)

print("Selected-model output root:", HYBRID_ROOT)


In [ ]:

# ============================================================
# 8. Run the selected parameter combination and save overlays
# ============================================================
all_parameter_scene_rows = []
RESULTS_BY_PARAMETER = {}

for combo_index, combo in enumerate(PARAMETER_COMBINATIONS, start=1):
    A = int(combo["A"])
    S = int(combo["S"])
    e = float(combo["e"])
    parameter_label = str(combo["label"])

    # Update the exact globals read by the grid-search rollout.
    ABSTRACTION_CHECK_CHUNK_FRAMES = A
    SIM_CHUNK_FRAMES = S
    ADAPTIVE_SIM_INCREMENT_FRAMES = S

    parameter_dir = HYBRID_ROOT / parameter_label
    prediction_dir = parameter_dir / "hybrid_predictions"
    overlay_dir = parameter_dir / "hybrid_overlay_plots"

    prediction_dir.mkdir(parents=True, exist_ok=True)
    overlay_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "=" * 100)
    print(
        f"Parameter set {combo_index}/{len(PARAMETER_COMBINATIONS)}: "
        f"A={A}, S={S}, e={e:.1f}"
    )
    print("Output:", parameter_dir.resolve())
    print("=" * 100)

    parameter_rows = []

    for scene_index, record in enumerate(SCENE_RECORDS, start=1):
        scene_dir = Path(record["scene_dir"])
        scene_name = str(record["scene_name"])

        print(
            f"[{parameter_label}] scene "
            f"{scene_index:03d}/{len(SCENE_RECORDS):03d}: {scene_name}",
            flush=True,
        )

        hybrid_df = run_hybrid_for_scene(
            scene_dir,
            threshold_pixels=e,
            max_segments=MAX_SEGMENTS_PER_SCENE,
        )

        abstraction_steps = count_abstraction_steps(hybrid_df)
        simulation_steps = count_simulation_steps(hybrid_df)
        total_steps = int(abstraction_steps + simulation_steps)

        hybrid_pred_hit = int(hybrid_pred_hit_from_df(hybrid_df))
        hybrid_correct = int(hybrid_pred_hit == int(record["true_hit"]))

        prediction_path = (
            prediction_dir
            / f"{safe_scene_id(scene_dir)}__hybrid_predictions.csv"
        )
        overlay_path = (
            overlay_dir
            / f"{safe_scene_id(scene_dir)}__hybrid_overlay.png"
        )

        hybrid_df.to_csv(prediction_path, index=False)
        plot_hybrid_overlay(scene_dir, hybrid_df, overlay_path)

        row = {
            "parameter_label": parameter_label,
            "A": A,
            "S": S,
            "e": e,
            "scene_name": scene_name,
            "scene_dir": str(scene_dir),
            "simulation_time": float(record["simulation_time"]),
            "true_hit": int(record["true_hit"]),
            "hybrid_pred_hit": hybrid_pred_hit,
            "hybrid_correct": hybrid_correct,
            "abstraction_step_count": abstraction_steps,
            "simulation_step_count": simulation_steps,
            "hybrid_total_step_count": total_steps,
            "prediction_csv": str(prediction_path),
            "overlay_png": str(overlay_path),
            **summarize_hybrid_df(hybrid_df),
        }

        parameter_rows.append(row)
        all_parameter_scene_rows.append(row)

    parameter_results = pd.DataFrame(parameter_rows)
    parameter_results.to_csv(
        parameter_dir / "hybrid_scene_summary.csv",
        index=False,
    )
    RESULTS_BY_PARAMETER[parameter_label] = parameter_results

    print(
        f"Completed {parameter_label}: "
        f"accuracy={parameter_results['hybrid_correct'].mean():.4f}"
    )
    print("Overlays:", overlay_dir.resolve())

all_results_df = pd.DataFrame(all_parameter_scene_rows)
all_results_df.to_csv(
    HYBRID_ROOT / "combined_hybrid_scene_summary.csv",
    index=False,
)

expected_rows = len(PARAMETER_COMBINATIONS) * len(SCENE_RECORDS)
if len(all_results_df) != expected_rows:
    raise RuntimeError(
        f"Expected {expected_rows} completed scene runs, "
        f"found {len(all_results_df)}."
    )

display(
    all_results_df.groupby(["parameter_label", "A", "S", "e"]).agg(
        n_scenes=("scene_name", "size"),
        mean_accuracy=("hybrid_correct", "mean"),
        mean_total_steps=("hybrid_total_step_count", "mean"),
        median_total_steps=("hybrid_total_step_count", "median"),
    ).reset_index()
)

print("All saved model outputs (earlier contents overwritten):", HYBRID_ROOT.resolve())
